# Forecasting Formula 1 Performance and Race Outcomes with Machine Learning
**Group E** | Introduction to Machine Learning | BS Analytics 2024-2028

---

## How to run this notebook

Run the cells **in the order they appear**. They are numbered in the section headings below.

The notebook has two halves:

| Part | Cells | Needs |
|---|---|---|
| **A. Data collection** (Sections 1-3) | 1 to 8 | An F1TV login, internet, and about an hour. **Run once.** The data it produces is already in the `Data` folder and is submitted with this notebook. |
| **B. Analysis and models** (Sections 4-9) | 9 to 22 | Nothing but the saved files. This is the part to run for marking. |

**The one manual step** is the F1TV login in Cell 2. FastF1 has required an account
since October 2026 and limits requests to 500 per hour. Nothing else needs manual input.

**To reproduce the reported results:** run Cell 1, then skip to Section 4 and run
Cells 9 to 22 in order. Every table and figure in the report comes from those cells.

Each model is compared against a simple baseline. Cells marked **v1** are the first
version of a model; they are kept because the report uses them as comparison rows,
and because Cell 16 reads a file that Cell 15 writes.

## Pipeline
The notebook has 23 numbered cells in 8 sections. Run them in the order they appear.

| Stage | Section | Cells | What happens | Reads | Writes |
|---|---|---|---|---|---|
| Setup | 1 | 1 | Sets the project folder, defines `find()`, and fixes one time-based split for every model (train R1-12, validate R13-16, test R17-20) | none | `BASE`, `find()`, `split()` |
| A. Data collection (run once) | 2 | 2-8 | Installs FastF1, logs in to F1TV, pulls 2026 race laps and results, qualifying, practice/sprint/pit/race-control data, and 2022-25 results. Resumable if the hourly API limit is hit | FastF1 | `F1_2026_races.xlsx`, `F1_2026_qualifying.xlsx`, `F1_2026_extra.xlsx`, `hist_results.parquet` |
| B. Modelling tables | 3 | 9 | Rebuilds the lap-level pace and pit tables from the three workbooks, with preprocessing and outlier cutoffs from rounds 1-16 only | the 3 workbooks, stored `model_pace.parquet` | `model_pace.parquet`, `model_pit.parquet` (only if `OVERWRITE = True`) |
| C. Does history help? | 4 | 10-12 | Tests whether 2022-25 data predicts 2026: driver-at-circuit affinity (10), circuit behaviour (11), and driver rating from team-mate qualifying gap (12). Only the driver rating is kept | `hist_results.parquet`, `model_race.parquet`, qualifying workbook | `hist_driver_affinity.parquet`, `hist_circuit.parquet` |
| D. In-race models | 5 | 13-15 | Model 1 lap pace and Model 2 tyre degradation (13), Model 3 pit next lap (14), Model 8 stint consistency (15). Each is compared with a baseline | `model_pace.parquet`, `model_pit.parquet` | `model1_pace`, `model3_pit`, `model8_consistency`, results CSVs |
| E. Forecast models, v1 | 6 | 16-17 | Model 5 Q1 gap (16) and Models 6 and 7 regression (17). Kept as comparison rows | `model_quali.parquet`, `model_race.parquet` | `model5_quali`, `model6_finish`, `model7_points`, results CSVs |
| F. Forecast models, v2 | 6 | 18-19 | Model 5 qualifying order (18) and Models 6 and 7 Plackett-Luce ranking with 20,000 simulated races (19). Walk-forward over rounds 8-20 | `model_quali.parquet`, `model_race.parquet`, `hist_results.parquet`, `model6_finish.json` | `model5_quali_form/fp.pkl`, `model6_pl_pre/post.pkl`, `site_export/walkforward_*.csv` |
| G. Forecast remaining races | 7 | 20-22 | Three-stage forecast for rounds 21-23: before the weekend, after practice, after qualifying (20). Seal the Stage 1 forecast with a hash (21). Score it once the race has run (22) | `.pkl` models from 18 and 19, FastF1 (stages 2-3) | `forecast_stage1.csv`, `forecast_R*_stage*.csv`, `seal_stage1_v4.json` |
| H. Live-race replay | 8 | 23 | Replays a test race (R17-20) lap by lap through Models 1, 2 and 3 | `model1_pace`, `model3_pit`, `model_pace`, `model_pit` | `site_export/replay_*` |

## How models are compared
Every model is scored on races it never saw. The split is time-based, and walk-forward means race *r* is predicted using only races before *r*. Each model is compared with a simple baseline (the training average, *finish = grid position*, or *never pit*). A model only counts if it beats its baseline, and the model is chosen on validation, not test.

## Section 1. Setup

### Cell 1. Project folder and shared helpers
Run this first, every time the notebook is opened. It sets the working folder and defines `find()`, which locates a data file in either the project folder or the `Data` subfolder.

# Libraries

In [ ]:
!pip install openpyxl

import os, json, glob, warnings
import numpy as np, pandas as pd, joblib
warnings.filterwarnings("ignore")
import getpass
import fastf1
import fastf1.plotting
import matplotlib.pyplot as plt
import time
from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_score,recall_score, f1_score, confusion_matrix, accuracy_score)
from scipy.optimize import minimize

In [ ]:
# ============================================================
# CELL 0 - SETUP (run this first, every time you open the notebook)
# Points the notebook at your project folder. Everything is read
# from and saved to this folder.
# ============================================================

BASE = os.path.join(os.path.expanduser("~"), "OneDrive", "Desktop", "Yug", "Sem 5", "ML", "Grp assignment")
if not os.path.isdir(BASE):
    raise FileNotFoundError("Folder not found. In File Explorer, click the address bar of 'Grp assignment', "
                            "copy the path, and paste it as BASE = r\"<path>\"")
os.chdir(BASE)                                   # everything the models create is saved here
os.makedirs("saved_models", exist_ok=True)
os.makedirs("figures", exist_ok=True)

def find(name):                                  # looks in Grp assignment, then in Grp assignment\Data
    for d in [BASE, os.path.join(BASE, "Data")]:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"{name} not found in {BASE} or {os.path.join(BASE, 'Data')}")

RACE_FILE, QUALI_FILE, EXTRA_FILE = find("F1_2026_races.xlsx"), find("F1_2026_qualifying.xlsx"), find("F1_2026_extra.xlsx")
for f in [RACE_FILE, QUALI_FILE, EXTRA_FILE]:
    print(f"{os.path.basename(f):28s} found: {os.path.exists(f)}  ({os.path.dirname(f)})")

# one split for every model: earlier races train, later races test
TRAIN_ROUNDS = range(1, 13)    # rounds 1-12  : fit
VALID_ROUNDS = range(13, 17)   # rounds 13-16 : choose model / threshold
TEST_ROUNDS  = range(17, 21)   # rounds 17-20 : final score, touched once
def split(df, col="Round"):
    return (df[df[col].isin(TRAIN_ROUNDS)], df[df[col].isin(VALID_ROUNDS)], df[df[col].isin(TEST_ROUNDS)])

class Baseline:                 # predicts the training average for everyone
    def fit(self, X, y): self.m = float(np.mean(y)); return self
    def predict(self, X): return np.full(len(X), self.m)

print("\nWorking folder (outputs go here):", os.getcwd())

F1_2026_races.xlsx           found: True  (C:\Users\bhogi\OneDrive\Desktop\Yug\Sem 5\ML\Grp assignment\Data)
F1_2026_qualifying.xlsx      found: True  (C:\Users\bhogi\OneDrive\Desktop\Yug\Sem 5\ML\Grp assignment\Data)
F1_2026_extra.xlsx           found: True  (C:\Users\bhogi\OneDrive\Desktop\Yug\Sem 5\ML\Grp assignment\Data)

Working folder (outputs go here): C:\Users\bhogi\OneDrive\Desktop\Yug\Sem 5\ML\Grp assignment


## Section 2. Data collection (run once)

These cells download the season from FastF1 and write the three Excel workbooks.
They need an F1TV login and are subject to a 500-request hourly limit, so the pull
loop is resumable: if it stops, re-run the same cell and it continues where it left off.

**Skip this section when marking.** The workbooks are in the `Data` folder.

### Cell 2. Install FastF1

In [ ]:
!pip install --upgrade fastf1

### Cell 3. F1TV login
Only needed the first time. Every cell skips itself when its output file already exists. The account details are typed at the prompt and are never written into the notebook.

In [ ]:
# Run ONLY if you need to download data (skip if all four data files already exist).
os.environ["F1_USERNAME"] = input("F1 account email: ")
os.environ["F1_PASSWORD"] = getpass.getpass("F1 account password: ")

!python -m fastf1 auth f1tv --authenticate
!python -m fastf1 auth f1tv --status

F1 account email:  bhogi.yug@gmail.com
F1 account password:  ········


Sign-in successful.


tokens         INFO 	Signing in to Formula 1 account...
req         WARNING 	DEFAULT CACHE ENABLED! (24.0 KB) C:\Users\bhogi\AppData\Local\Temp\fastf1
tokens         INFO 	Sign-in successful.


Sign-in method: password
Token Status: Expires 2026-11-14 15:34:20
Subscription Status: inactive
Subscribed Product: 


### Cell 4. Open a FastF1 session and check the version

In [ ]:

print(fastf1.__version__)  # should be 3.9.0 or newer

os.makedirs("f1_cache", exist_ok=True)
fastf1.Cache.enable_cache("f1_cache")  # must be called before loading any session

3.9.2


C:\Users\bhogi\anaconda3\Lib\site-packages\fastf1\req.py:251: FutureWarning: `.enable_cache` is deprecated and will be removed in a future version. Use `.configure` instead.
  warnings.warn("`.enable_cache` is deprecated and will be removed in a "


### Cell 5. Pull race laps and results

In [ ]:

os.makedirs("data", exist_ok=True)
WEATHER_COLS = ["AirTemp", "TrackTemp", "Humidity", "Pressure",
                "WindSpeed", "WindDirection", "Rainfall"]

def clean_for_excel(df):
    # timedeltas become plain seconds, timezone-aware dates lose the timezone
    for col in list(df.columns):
        if pd.api.types.is_timedelta64_dtype(df[col]):
            df[col + "_sec"] = df[col].dt.total_seconds()
            df = df.drop(columns=[col])
        elif isinstance(df[col].dtype, pd.DatetimeTZDtype):
            df[col] = df[col].dt.tz_localize(None)
    return df

def pull_race(rnd):
    p_laps = f"data/laps_2026_{rnd:02d}.csv"
    p_res = f"data/results_2026_{rnd:02d}.csv"
    if os.path.exists(p_laps) and os.path.exists(p_res):
        return "skipped, already saved"
    try:
        s = fastf1.get_session(2026, rnd, "R")
        s.load(laps=True, telemetry=False, weather=True, messages=False)

        w = s.laps.get_weather_data().reset_index(drop=True)[WEATHER_COLS]
        laps = s.laps.reset_index(drop=True)
        laps = pd.concat([laps, w], axis=1)
        res = s.results.reset_index(drop=True).copy()

        for df in (laps, res):
            df["Round"] = rnd
            df["EventName"] = s.event["EventName"]
            df["Location"] = s.event["Location"]
            df["Country"] = s.event["Country"]

        clean_for_excel(laps).to_csv(p_laps, index=False)
        clean_for_excel(res).to_csv(p_res, index=False)
        return f"ok ({len(laps)} laps)"
    except Exception as e:
        return f"FAILED: {type(e).__name__}: {e}"

schedule = fastf1.get_event_schedule(2026, include_testing=False)
schedule = schedule[schedule["EventDate"] < pd.Timestamp.now()]
print(f"{len(schedule)} races found so far in 2026")

log = []
for _, row in schedule.iterrows():
    rnd = int(row["RoundNumber"])
    status = pull_race(rnd)
    log.append((rnd, row["EventName"], status))
    print(rnd, row["EventName"], status)
    time.sleep(1)

pd.DataFrame(log, columns=["round", "event", "status"]).to_csv("data/pull_log_2026.csv", index=False)

core           INFO 	Loading data for Australian Grand Prix - Race [v3.9.2]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


20 races found so far in 2026


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!

1 Australian Grand Prix ok (1006 laps)


core           INFO 	Loading data for Chinese Grand Prix - Race [v3.9.2]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No ca

2 Chinese Grand Prix ok (920 laps)


... (32 outputs left out)

core           INFO 	Loading data for Mexico City Grand Prix - Race [v3.9.2]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	N

19 Mexico City Grand Prix ok (1103 laps)


core           INFO 	Loading data for São Paulo Grand Prix - Race [v3.9.2]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No 

20 São Paulo Grand Prix ok (1435 laps)


### Cell 6. Pull qualifying

In [ ]:

laps = pd.concat([pd.read_csv(f) for f in sorted(glob.glob("data/laps_2026_*.csv"))], ignore_index=True)
results = pd.concat([pd.read_csv(f) for f in sorted(glob.glob("data/results_2026_*.csv"))], ignore_index=True)
log_df = pd.read_csv("data/pull_log_2026.csv")

with pd.ExcelWriter("F1_2026_races.xlsx", engine="openpyxl") as writer:
    laps.to_excel(writer, sheet_name="Laps", index=False)
    results.to_excel(writer, sheet_name="Results", index=False)
    log_df.to_excel(writer, sheet_name="Pull_log", index=False)

print("Laps:", laps.shape)
print("Results:", results.shape)
print("Races in file:", laps["Round"].nunique())
print(laps.groupby("EventName")["Driver"].count())

Laps: (23323, 42)
Results: (440, 26)
Races in file: 20
EventName
Australian Grand Prix       1006
Austrian Grand Prix         1339
Azerbaijan Grand Prix        981
Bahrain Grand Prix          1145
Barcelona Grand Prix        1236
Belgian Grand Prix           872
British Grand Prix          1113
Canadian Grand Prix         1211
Chinese Grand Prix           920
Dutch Grand Prix            1368
Hungarian Grand Prix        1431
Italian Grand Prix          1054
Japanese Grand Prix         1107
Mexico City Grand Prix      1103
Miami Grand Prix            1040
Monaco Grand Prix           1452
Singapore Grand Prix        1240
Spanish Grand Prix          1106
São Paulo Grand Prix        1435
United States Grand Prix    1164
Name: Driver, dtype: int64


In [ ]:

os.makedirs("data", exist_ok=True)
WEATHER_COLS = ["AirTemp", "TrackTemp", "Humidity", "Pressure",
                "WindSpeed", "WindDirection", "Rainfall"]

def clean_for_excel(df):
    for col in list(df.columns):
        if pd.api.types.is_timedelta64_dtype(df[col]):
            df[col + "_sec"] = df[col].dt.total_seconds()
            df = df.drop(columns=[col])
        elif isinstance(df[col].dtype, pd.DatetimeTZDtype):
            df[col] = df[col].dt.tz_localize(None)
    return df

def pull_quali(rnd):
    p_laps = f"data/qlaps_2026_{rnd:02d}.csv"
    p_res = f"data/qres_2026_{rnd:02d}.csv"
    if os.path.exists(p_laps) and os.path.exists(p_res):
        return "skipped, already saved"
    try:
        s = fastf1.get_session(2026, rnd, "Q")
        s.load(laps=True, telemetry=False, weather=True, messages=False)
        w = s.laps.get_weather_data().reset_index(drop=True)[WEATHER_COLS]
        laps = pd.concat([s.laps.reset_index(drop=True), w], axis=1)
        res = s.results.reset_index(drop=True).copy()
        for df in (laps, res):
            df["Round"] = rnd
            df["EventName"] = s.event["EventName"]
            df["Location"] = s.event["Location"]
        clean_for_excel(laps).to_csv(p_laps, index=False)
        clean_for_excel(res).to_csv(p_res, index=False)
        return f"ok ({len(laps)} laps)"
    except Exception as e:
        return f"FAILED: {type(e).__name__}: {e}"

schedule = fastf1.get_event_schedule(2026, include_testing=False)
schedule = schedule[schedule["EventDate"] < pd.Timestamp.now()]

for _, row in schedule.iterrows():
    rnd = int(row["RoundNumber"])
    print(rnd, row["EventName"], pull_quali(rnd))
    time.sleep(1)

qlaps = pd.concat([pd.read_csv(f) for f in sorted(glob.glob("data/qlaps_2026_*.csv"))], ignore_index=True)
qres = pd.concat([pd.read_csv(f) for f in sorted(glob.glob("data/qres_2026_*.csv"))], ignore_index=True)

with pd.ExcelWriter("F1_2026_qualifying.xlsx", engine="openpyxl") as writer:
    qlaps.to_excel(writer, sheet_name="Quali_Laps", index=False)
    qres.to_excel(writer, sheet_name="Quali_Results", index=False)

print("Quali laps:", qlaps.shape, "| Quali results:", qres.shape)
print("Q3 times filled:", qres["Q3_sec"].notna().sum(), "of", len(qres))

1 Australian Grand Prix skipped, already saved
2 Chinese Grand Prix skipped, already saved
3 Japanese Grand Prix skipped, already saved
4 Miami Grand Prix skipped, already saved
5 Canadian Grand Prix skipped, already saved
6 Monaco Grand Prix skipped, already saved
7 Barcelona Grand Prix skipped, already saved
8 Austrian Grand Prix skipped, already saved
9 British Grand Prix skipped, already saved
10 Belgian Grand Prix skipped, already saved
11 Hungarian Grand Prix skipped, already saved
12 Dutch Grand Prix skipped, already saved
13 Italian Grand Prix skipped, already saved
14 Spanish Grand Prix skipped, already saved
15 Azerbaijan Grand Prix skipped, already saved
16 Bahrain Grand Prix skipped, already saved
17 Singapore Grand Prix skipped, already saved
18 United States Grand Prix skipped, already saved
19 Mexico City Grand Prix skipped, already saved
20 São Paulo Grand Prix skipped, already saved
Quali laps: (6693, 41) | Quali results: (440, 25)
Q3 times filled: 195 of 440


### Cell 7. Pull practice, sprint, pit stops, race control and track status
Writes `F1_2026_extra.xlsx`, the third workbook.

In [ ]:
# ============================================================
# STEP 5 - EXTRA DATA PULL  ->  F1_2026_extra.xlsx  (3rd Excel file)
# Run the LOGIN cell first. Your F1 token expires 14 Nov.
# Pulls: practice (FP1-3), sprint qualifying + sprint, race control
# messages (race, quali, sprint), exact safety car timeline, pit
# stops, circuit corners, Pirelli C-compound per race, circuit type.
# Re-runnable: anything already saved is skipped.
# ============================================================
DL = os.path.expanduser("~/Downloads")
race_file = os.path.join(DL, "F1_2026_races.xlsx")
os.makedirs("data_extra", exist_ok=True)
fastf1.Cache.enable_cache("f1_cache")

WEATHER_COLS = ["AirTemp","TrackTemp","Humidity","Pressure","WindSpeed","WindDirection","Rainfall"]
SESSION_CODES = {"Practice 1":"FP1", "Practice 2":"FP2", "Practice 3":"FP3",
                 "Sprint Qualifying":"SQ", "Sprint Shootout":"SQ", "Sprint":"S"}

# ---------- Pirelli compounds per race (checked against Pirelli / F1 announcements) ----------
# C1 = hardest ... C5 = softest. 2026 has only 5 compounds.
PIRELLI = {  # round: (event name to check, Hard, Medium, Soft)
 1:("Australian",3,4,5),  2:("Chinese",2,3,4),   3:("Japanese",1,2,3),  4:("Miami",3,4,5),
 5:("Canadian",3,4,5),    6:("Monaco",3,4,5),    7:("Barcelona",2,3,4), 8:("Austrian",3,4,5),
 9:("British",1,2,3),    10:("Belgian",2,3,4),  11:("Hungarian",3,4,5),12:("Dutch",2,3,4),
13:("Italian",3,4,5),    14:("Spanish",2,3,4),  15:("Azerbaijan",3,4,5),16:("Bahrain",2,3,4),
17:("Singapore",3,4,5),  18:("United States",2,3,4), 19:("Mexico",3,4,5), 20:("São Paulo",3,4,5),
}
# street = temporary city streets | park = temporary park road | permanent = purpose-built
CIRCUIT_TYPE = {"Monaco":"street","Azerbaijan":"street","Singapore":"street","Spanish":"street",
                "Miami":"street","Las Vegas":"street","Australian":"park","Canadian":"park"}

def clean_for_excel(df):
    df = df.copy()
    for col in list(df.columns):
        if pd.api.types.is_timedelta64_dtype(df[col]):
            df[col + "_sec"] = df[col].dt.total_seconds()
            df = df.drop(columns=[col])
        elif isinstance(df[col].dtype, pd.DatetimeTZDtype):
            df[col] = df[col].dt.tz_localize(None)
    return df

def tag(df, rnd, event, code):
    df = df.copy(); df["Round"] = rnd; df["EventName"] = event; df["Session"] = code
    return df

# ---------- 1. practice + sprint sessions: laps, weather, messages, sprint results ----------
def pull_lap_session(rnd, event, name, code):
    p_laps = f"data_extra/laps_{rnd:02d}_{code}.csv"
    if os.path.exists(p_laps):
        return "skipped, already saved"
    try:
        s = fastf1.get_session(2026, rnd, name)
        s.load(laps=True, telemetry=False, weather=True, messages=True)
        w = s.laps.get_weather_data().reset_index(drop=True)[WEATHER_COLS]
        laps = pd.concat([s.laps.reset_index(drop=True), w], axis=1)
        clean_for_excel(tag(laps, rnd, event, code)).to_csv(p_laps, index=False)
        clean_for_excel(tag(s.race_control_messages, rnd, event, code)).to_csv(
            f"data_extra/rc_{rnd:02d}_{code}.csv", index=False)
        if code == "S":
            clean_for_excel(tag(s.results, rnd, event, code)).to_csv(
                f"data_extra/sres_{rnd:02d}.csv", index=False)
        return f"ok ({len(laps)} laps)"
    except Exception as e:
        return f"FAILED: {type(e).__name__}: {e}"

# ---------- 2. race + quali extras: messages, safety car timeline, corners ----------
def pull_race_quali_extras(rnd, event, code):
    p_rc = f"data_extra/rc_{rnd:02d}_{code}.csv"
    if os.path.exists(p_rc):
        return "skipped, already saved", None
    try:
        s = fastf1.get_session(2026, rnd, code)
        s.load(laps=True, telemetry=False, weather=False, messages=True)   # laps come from cache
        clean_for_excel(tag(s.race_control_messages, rnd, event, code)).to_csv(p_rc, index=False)
        corners = np.nan
        if code == "R":
            clean_for_excel(tag(s.track_status, rnd, event, code)).to_csv(
                f"data_extra/ts_{rnd:02d}.csv", index=False)
            try:
                corners = len(s.get_circuit_info().corners)
            except Exception:
                pass
        return "ok", corners
    except Exception as e:
        return f"FAILED: {type(e).__name__}: {e}", None

schedule = fastf1.get_event_schedule(2026, include_testing=False)
schedule = schedule[schedule["EventDate"] < pd.Timestamp.now()]
print(f"{len(schedule)} events to process\n")

# ---- restore corners from earlier runs so a rerun doesn't lose them ----
import json
corner_file = "data_extra/corners.json"
if os.path.exists(corner_file):
    corners_by_round = {int(k): v for k, v in json.load(open(corner_file)).items()}
else:
    corners_by_round = dict(globals().get("corners_by_round", {}))   # keeps corners from your first run
log = []
stop = False

def hit_limit(status):
    return "RateLimit" in str(status)

for _, ev in schedule.iterrows():
    if stop:
        break
    rnd, event = int(ev["RoundNumber"]), ev["EventName"]
    for i in range(1, 6):                                     # Session1..Session5 of the weekend
        name = ev.get(f"Session{i}")
        if name in SESSION_CODES:
            status = pull_lap_session(rnd, event, name, SESSION_CODES[name])
            log.append((rnd, event, SESSION_CODES[name], status)); print(rnd, event, SESSION_CODES[name], status)
            time.sleep(1)
            if hit_limit(status):
                stop = True
                break
    if stop:
        break
    for code in ["Q", "R"]:
        status, corners = pull_race_quali_extras(rnd, event, code)
        if corners is not None and not pd.isna(corners):
            corners_by_round[rnd] = corners
        log.append((rnd, event, code + " extras", status)); print(rnd, event, code, "extras", status)
        time.sleep(1)
        if hit_limit(status):
            stop = True
            break

json.dump(corners_by_round, open(corner_file, "w"))
if stop:
    print("\n>>> Hourly API limit hit. Wait about 1 hour, then rerun this cell. Saved rounds are skipped.")
else:
    print("\n>>> All sessions processed.")

# ---------- 3. pit stops from your existing race file ----------
rlaps = pd.read_excel(race_file, sheet_name="Laps").sort_values(["Round","Driver","LapNumber"])
rlaps["next_out"] = rlaps.groupby(["Round","Driver"])["PitOutTime_sec"].shift(-1)
pits = rlaps[rlaps["PitInTime_sec"].notna() & rlaps["next_out"].notna()].copy()
pits["pit_lane_sec"] = pits["next_out"] - pits["PitInTime_sec"]
pits["red_flag_or_long"] = (pits["pit_lane_sec"] > 60).astype(int)   # red flag / repairs, not a normal stop
pits["compound_before"] = pits["Compound"]
pits["compound_after"] = rlaps.groupby(["Round","Driver"])["Compound"].shift(-1).loc[pits.index]
pits = pits[["Round","EventName","Driver","Team","LapNumber","TyreLife","compound_before",
             "compound_after","pit_lane_sec","red_flag_or_long"]]

# ---------- 4. one row per event: circuit + tyre facts ----------
info = schedule[["RoundNumber","EventName","Location","Country","EventDate","EventFormat"]].rename(
        columns={"RoundNumber":"Round"}).copy()
info["race_laps"] = info["Round"].map(rlaps.groupby("Round")["LapNumber"].max())
info["n_corners"] = info["Round"].map(corners_by_round)
info["pit_lane_median_sec"] = info["Round"].map(pits[pits.red_flag_or_long == 0].groupby("Round")["pit_lane_sec"].median())
info["circuit_type"] = info["EventName"].apply(
        lambda n: next((t for k, t in CIRCUIT_TYPE.items() if k in n), "permanent"))
info["is_sprint_weekend"] = info["EventFormat"].astype(str).str.contains("sprint").astype(int)
for i, col in enumerate(["hard_C","medium_C","soft_C"], start=1):
    info[col] = info["Round"].map({r: v[i] for r, v in PIRELLI.items()})
info["pirelli_name_check"] = info.apply(
        lambda r: "OK" if r["Round"] in PIRELLI and PIRELLI[r["Round"]][0] in r["EventName"] else "CHECK", axis=1)

# ---------- 5. combine everything into the 3rd Excel file ----------
def read_all(pattern):
    files = sorted(glob.glob(pattern))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame()

def add_c_number(df):   # "SOFT" at Monaco is C5, at Suzuka it's C3: give every lap its real compound
    if df.empty or "Compound" not in df: return df
    m = info.set_index("Round")[["hard_C","medium_C","soft_C"]]
    pick = {"HARD":"hard_C","MEDIUM":"medium_C","SOFT":"soft_C"}
    df["compound_C"] = [m.at[r, pick[c]] if (c in pick and r in m.index) else np.nan
                        for r, c in zip(df["Round"], df["Compound"])]
    return df

practice = add_c_number(read_all("data_extra/laps_*_FP*.csv"))
sprint   = add_c_number(pd.concat([read_all("data_extra/laps_*_SQ.csv"), read_all("data_extra/laps_*_S.csv")], ignore_index=True))
sres     = read_all("data_extra/sres_*.csv")
rc       = read_all("data_extra/rc_*.csv")
ts       = read_all("data_extra/ts_*.csv")
log_df   = pd.DataFrame(log, columns=["round","event","session","status"])

out = os.path.join(DL, "F1_2026_extra.xlsx")
with pd.ExcelWriter(out, engine="openpyxl") as w:
    info.to_excel(w, sheet_name="Event_Info", index=False)
    practice.to_excel(w, sheet_name="Practice_Laps", index=False)
    sprint.to_excel(w, sheet_name="Sprint_Laps", index=False)
    sres.to_excel(w, sheet_name="Sprint_Results", index=False)
    pits.to_excel(w, sheet_name="Pit_Stops", index=False)
    rc.to_excel(w, sheet_name="Race_Control", index=False)
    ts.to_excel(w, sheet_name="Track_Status", index=False)
    log_df.to_excel(w, sheet_name="Pull_log", index=False)

# ---------- 6. checks ----------
print("\nSaved:", out)
for nm, d in [("Practice_Laps",practice),("Sprint_Laps",sprint),("Sprint_Results",sres),
              ("Pit_Stops",pits),("Race_Control",rc),("Track_Status",ts)]:
    print(f"{nm:15s} {d.shape}")
print("\nFAILED pulls:", (log_df.status.str.startswith("FAILED")).sum())
print("Pirelli name check:", info.pirelli_name_check.value_counts().to_dict())
print("\nEvent_Info preview:")
print(info[["Round","EventName","EventFormat","circuit_type","n_corners","pit_lane_median_sec",
            "hard_C","medium_C","soft_C"]].to_string(index=False))

20 events to process

1 Australian Grand Prix FP1 skipped, already saved
1 Australian Grand Prix FP2 skipped, already saved
1 Australian Grand Prix FP3 skipped, already saved
1 Australian Grand Prix Q extras skipped, already saved
1 Australian Grand Prix R extras skipped, already saved
2 Chinese Grand Prix FP1 skipped, already saved
2 Chinese Grand Prix SQ skipped, already saved
2 Chinese Grand Prix S skipped, already saved
2 Chinese Grand Prix Q extras skipped, already saved
2 Chinese Grand Prix R extras skipped, already saved
3 Japanese Grand Prix FP1 skipped, already saved
3 Japanese Grand Prix FP2 skipped, already saved
3 Japanese Grand Prix FP3 skipped, already saved
3 Japanese Grand Prix Q extras skipped, already saved
3 Japanese Grand Prix R extras skipped, already saved
4 Miami Grand Prix FP1 skipped, already saved
4 Miami Grand Prix SQ skipped, already saved
4 Miami Grand Prix S skipped, already saved
4 Miami Grand Prix Q extras skipped, already saved
4 Miami Grand Prix R extr

core           INFO 	Loading data for Singapore Grand Prix - Practice 1 [v3.9.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	No cached data found for timing_app_data. Loading data...
_api           INFO 	Fetching timing app data...
req            INFO 	Data has been written to cache!
core           INFO 	Processing timing data...
req            INFO 	No cached data found for weather_data. Loading data...
_api           INFO 	Fetching weather data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for race_control_messages. Loading data...
_api           INFO 	Fetching race control messages...
req            INFO 	Data has been written to cache!
core           INFO 	Fi

17 Singapore Grand Prix FP1 ok (568 laps)


core           INFO 	Loading data for Singapore Grand Prix - Sprint Qualifying [v3.9.2]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
core        WARNING 	Sprint Qualifying is not supported by Ergast! Limited results are calculated from timing data.
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. 

17 Singapore Grand Prix SQ ok (286 laps)


core           INFO 	Loading data for Singapore Grand Prix - Sprint [v3.9.2]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	N

... (31 outputs left out)

core           INFO 	Loading data for São Paulo Grand Prix - Qualifying [v3.9.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	No cached data found for race_control_messages. Loading data...
_api           INFO 	Fetching race control messages...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 22 drivers: ['1', '63', '12', '16', '30', '27', '6', '87', '14', '23', '44', '81', '51', '41', '55', '3', '43', '10', '18', '77', '11', '5']


20 São Paulo Grand Prix Q extras ok


core           INFO 	Loading data for São Paulo Grand Prix - Race [v3.9.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '18'
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 77)
req            INFO 	No cached data found for race_control_messages. Loading data...
_api           INFO 	Fetching race control messages...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 22 drivers: ['1', '3', '6', '63', '12', '81', '3

20 São Paulo Grand Prix R extras ok

>>> All sessions processed.

Saved: C:\Users\bhogi/Downloads\F1_2026_extra.xlsx
Practice_Laps   (24708, 42)
Sprint_Laps     (4030, 42)
Sprint_Results  (132, 25)
Pit_Stops       (833, 10)
Race_Control    (8948, 12)
Track_Status    (394, 6)

FAILED pulls: 1
Pirelli name check: {'OK': 20}

Event_Info preview:
 Round                EventName       EventFormat circuit_type  n_corners  pit_lane_median_sec  hard_C  medium_C  soft_C
     1    Australian Grand Prix      conventional         park       14.0              19.2295       3         4       5
     2       Chinese Grand Prix sprint_qualifying    permanent       16.0              23.3770       2         3       4
     3      Japanese Grand Prix      conventional    permanent       18.0              24.2670       1         2       3
     4         Miami Grand Prix sprint_qualifying       street       19.0              23.1520       3         4       5
     5      Canadian Grand Prix sprint_qualifying 

### Cell 8. Pull 2022-2025 results for the historical tests
Results only, no lap data. Writes `hist_results.parquet`.

In [ ]:
# ============================================================
# CELL 9A - HISTORICAL PULL 2022-2025 (results only, no laps)
# Gets qualifying + race results for every past race, so we can
# measure driver-at-circuit skill and circuit behaviour.
# Light: no laps, no telemetry, no weather, no messages.
# Resumable: rerun after an hour if the API limit stops it.
# Run the LOGIN cell first if your token has expired.
# ============================================================

YEARS = [2022, 2023, 2024, 2025]
os.makedirs("data_hist", exist_ok=True)
fastf1.Cache.enable_cache("f1_cache")

KEEP = ["Abbreviation","TeamName","GridPosition","Position","Points","Status","Q1","Q2","Q3"]

def pull(year, rnd, code):
    path = f"data_hist/{year}_{rnd:02d}_{code}.csv"
    if os.path.exists(path):
        return "skipped"
    try:
        s = fastf1.get_session(year, rnd, code)
        s.load(laps=False, telemetry=False, weather=False, messages=False)   # results only
        r = s.results.reset_index(drop=True)
        r = r[[c for c in KEEP if c in r.columns]].copy()
        for c in ["Q1","Q2","Q3"]:
            if c in r.columns:
                r[c + "_sec"] = pd.to_timedelta(r[c]).dt.total_seconds()
                r = r.drop(columns=[c])
        r["Year"], r["Round"], r["Session"] = year, rnd, code
        r["Location"] = s.event["Location"]
        r["EventName"] = s.event["EventName"]
        r.to_csv(path, index=False)
        return f"ok ({len(r)})"
    except Exception as e:
        return f"FAILED: {type(e).__name__}: {e}"

stop = False
for year in YEARS:
    if stop: break
    sched = fastf1.get_event_schedule(year, include_testing=False)
    sched = sched[sched["EventDate"] < pd.Timestamp.now()]
    for rnd in sched["RoundNumber"].astype(int):
        for code in ["Q", "R"]:
            st = pull(year, rnd, code)
            print(year, rnd, code, st)
            time.sleep(0.5)
            if "RateLimit" in st:
                stop = True; break
        if stop: break

n = len(glob.glob("data_hist/*.csv"))
print(f"\n{n} session files saved so far")
if stop:
    print(">>> Hourly API limit hit. Wait about an hour and rerun this cell. Saved files are skipped.")
else:
    hist = pd.concat([pd.read_csv(f) for f in sorted(glob.glob("data_hist/*.csv"))], ignore_index=True)
    hist.to_parquet("hist_results.parquet", index=False)
    print("Saved hist_results.parquet", hist.shape)
    print(hist.groupby(["Year","Session"]).Round.nunique().unstack().to_string())

C:\Users\bhogi\anaconda3\Lib\site-packages\fastf1\req.py:251: FutureWarning: `.enable_cache` is deprecated and will be removed in a future version. Use `.configure` instead.
  warnings.warn("`.enable_cache` is deprecated and will be removed in a "


2022 1 Q skipped
2022 1 R skipped
2022 2 Q skipped
2022 2 R skipped
2022 3 Q skipped
2022 3 R skipped
2022 4 Q skipped
2022 4 R skipped
2022 5 Q skipped
2022 5 R skipped
2022 6 Q skipped
2022 6 R skipped
2022 7 Q skipped
2022 7 R skipped
2022 8 Q skipped
2022 8 R skipped
2022 9 Q skipped
2022 9 R skipped
2022 10 Q skipped
2022 10 R skipped
2022 11 Q skipped
2022 11 R skipped
2022 12 Q skipped
2022 12 R skipped
2022 13 Q skipped
2022 13 R skipped
2022 14 Q skipped
2022 14 R skipped
2022 15 Q skipped
2022 15 R skipped
2022 16 Q skipped
2022 16 R skipped
2022 17 Q skipped
2022 17 R skipped
2022 18 Q skipped
2022 18 R skipped
2022 19 Q skipped
2022 19 R skipped
2022 20 Q skipped
2022 20 R skipped
2022 21 Q skipped
2022 21 R skipped
2022 22 Q skipped
2022 22 R skipped
2023 1 Q skipped
2023 1 R skipped
2023 2 Q skipped
2023 2 R skipped
2023 3 Q skipped
2023 3 R skipped
2023 4 Q skipped
2023 4 R skipped
2023 5 Q skipped
2023 5 R skipped
2023 6 Q skipped
2023 6 R skipped
2023 7 Q skipped
2023 

core           INFO 	Loading data for Australian Grand Prix - Qualifying [v3.9.2]
req            INFO 	Using cached data for session_info
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['4', '81', '1', '63', '22', '23', '16', '44', '10', '55', '6', '14', '18', '7', '5', '12', '27', '30', '31', '87']


2025 1 Q ok (20)


core           INFO 	Loading data for Australian Grand Prix - Race [v3.9.2]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['4', '1', '63', '12', '23', '18', '27', '16', '81', '44', '10', '22', '31', '87', '30', '5', '14', '55', '7', '6']


2025 1 R ok (20)


... (88 outputs left out)

core           INFO 	Loading data for Abu Dhabi Grand Prix - Qualifying [v3.9.2]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '81', '63', '16', '14', '5', '31', '6', '22', '87', '55', '30', '12', '18', '44', '23', '27', '10', '43']


2025 24 Q ok (20)


core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.9.2]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['1', '81', '4', '16', '63', '14', '31', '44', '27', '18', '5', '87', '55', '22', '12', '23', '6', '30', '10', '43']


2025 24 R ok (20)

184 session files saved so far
Saved hist_results.parquet (3677, 14)
Session   Q   R
Year           
2022     22  22
2023     22  22
2024     24  24
2025     24  24


## Section 3. Build the modelling tables

### Cell 9. Build `model_pace` and `model_pit`



In [ ]:
# ============================================================
# CELL 9 - BUILD THE MODELLING TABLES
#
# Reads the three Excel workbooks and rebuilds model_pace.parquet and
# model_pit.parquet. Applies the preprocessing described in Section 4 of
# the report: race distance from Event_Info, separate decoding of safety
# car and yellow flags, outlier cutoffs taken from rounds 1-16 only, the
# compound-number mapping, and the practice gap features.
#
# RECONSTRUCTION NOTE (see report Appendix B):
# This cell was rebuilt from the project's own output files after the
# original was lost. It is verified against them: every column except two in the
# pace and pit tables reproduces exactly, including
# lap_delta (the target for Models 1, 2 and 8) and every pit feature used
# by Model 3. Two practice features, team_longrun_gap_pct and team_fp_deg,
# could not be re-derived and are carried over from the stored tables so
# that reported results are unchanged. Set VERIFY = True to print the
# column-by-column comparison.
# ============================================================

VERIFY = True          # compare the rebuild against the stored tables
OVERWRITE = False      # True = replace model_pace/model_pit .parquet files

RACE, QUAL, EXTRA = find("F1_2026_races.xlsx"), find("F1_2026_qualifying.xlsx"), find("F1_2026_extra.xlsx")
ev = pd.read_excel(EXTRA, sheet_name="Event_Info")
EV = ["Round","race_laps","n_corners","pit_lane_median_sec","circuit_type",
      "is_sprint_weekend","hard_C","medium_C","soft_C"]

# ---------- rain risk, read out of the race-control messages ----------
rc = pd.read_excel(EXTRA, sheet_name="Race_Control")
r  = rc[rc.Message.astype(str).str.contains("RISK OF RAIN", case=False, na=False)].copy()
r["p"] = r.Message.astype(str).str.extract(r"IS\s+(\d+)\s*%", flags=re.I).astype(float)
rain = pd.DataFrame({"Round": sorted(ev.Round.unique())})
rain["quali_rain_risk"] = rain.Round.map(r[r.Session != "R"].groupby("Round").p.max()).fillna(0.0)
rain["race_rain_risk"]  = rain.Round.map(r[r.Session == "R"].groupby("Round").p.max()).fillna(0.0)

# ---------- practice reference laps: practice + sprint, green and accurate ----------
F = pd.concat([pd.read_excel(EXTRA, sheet_name="Practice_Laps"),
               pd.read_excel(EXTRA, sheet_name="Sprint_Laps")], ignore_index=True)
F = F[F.LapTime_sec.notna() & F.IsAccurate & (F.TrackStatus.astype(str) == "1")].copy()

tb = F.groupby(["Round","Team"]).LapTime_sec.min().rename("b").reset_index()
tb["ov"] = tb.groupby("Round").b.transform("min")
tb["team_fp_gap_pct"] = 100 * (tb.b - tb.ov) / tb.ov

db = F.groupby(["Round","Driver"]).LapTime_sec.min().rename("b").reset_index()
db["ov"] = db.groupby("Round").b.transform("min")
db["driver_fp_gap_pct"] = 100 * (db.b - db.ov) / db.ov

# the two features that could not be re-derived: take them from the stored table
_stored = pd.read_parquet(find("model_pace.parquet"))
_carry  = _stored.groupby(["Round","Team"])[["team_longrun_gap_pct","team_fp_deg"]].first().reset_index()
PR = tb[["Round","Team","team_fp_gap_pct"]].merge(_carry, on=["Round","Team"], how="outer")

# ---------- laps ----------
laps = pd.read_excel(RACE, sheet_name="Laps").sort_values(["Round","Driver","LapNumber"]).reset_index(drop=True)
laps = laps.merge(ev[EV], on="Round", how="left").merge(rain, on="Round", how="left")
laps["n_corners"] = laps.n_corners.fillna(ev.n_corners.median())

for comp, col in {"HARD":"hard_C", "MEDIUM":"medium_C", "SOFT":"soft_C"}.items():
    s = laps.Compound.astype(str).str.upper() == comp
    laps.loc[s, "compound_C"] = laps.loc[s, col]

laps["laps_left"] = laps.race_laps - laps.LapNumber          # field distance, not the driver's last lap
laps["frac_done"] = laps.LapNumber / laps.race_laps
ts = laps.TrackStatus.astype(str)
laps["sc_or_vsc"] = ts.str.contains("4|6|7", regex=True).astype(int)   # 4 = SC, 6/7 = VSC
laps["yellow"]    = ts.str.contains("2", regex=True).astype(int)       # 2 = yellow, a different thing
laps["is_green"]  = (ts == "1").astype(int)

laps["is_pit_lap"] = laps.PitInTime_sec.notna().astype(int)
g = laps.groupby(["Round","Driver"])
laps["pit_next_lap"]   = g.is_pit_lap.shift(-1)
laps["lap_delta_prev"] = g.LapTime_sec.diff()
laps["pace_trend_3"]   = g.lap_delta_prev.transform(lambda s: s.rolling(3, min_periods=1).mean())

# gaps to the car ahead and behind, on the same lap
laps = laps.sort_values(["Round","LapNumber","Time_sec"])
lg = laps.groupby(["Round","LapNumber"])
laps["gap_ahead"]  = lg.Time_sec.diff()
laps["gap_behind"] = lg.Time_sec.diff().shift(-1)
laps.loc[lg.cumcount(ascending=False) == 0, "gap_behind"] = np.nan
laps["dirty_air"] = (laps.gap_ahead < 1.5).fillna(False).astype(int)

tot = laps.groupby(["Round","LapNumber"]).is_pit_lap.transform("sum")
laps["cars_pitted_this_lap"] = tot - laps.is_pit_lap            # other cars, not himself
laps = laps.sort_values(["Round","Driver","LapNumber"])
laps["cars_pitted_prev2"] = (laps.groupby(["Round","Driver"]).cars_pitted_this_lap
                             .transform(lambda s: s.shift(1).rolling(2, min_periods=1).sum()).fillna(0))
laps = laps.merge(PR, on=["Round","Team"], how="left")

# ---------- pit table: every lap where we know what happened next ----------
pit = laps[laps.pit_next_lap.notna()].copy()
pit["pit_next_lap"] = pit.pit_next_lap.astype(int)

# ---------- pace table: clean, dry, green racing laps only ----------
pace = laps[laps.LapTime_sec.notna() & laps.IsAccurate & (laps.is_green == 1)
            & laps.PitInTime_sec.isna() & laps.PitOutTime_sec.isna()
            & laps.Compound.notna() & laps.TyreLife.notna()].copy()
pace["lap_delta"] = pace.LapTime_sec - pace.groupby(["Round","Driver"]).LapTime_sec.transform("median")
lo, hi = pace.loc[pace.Round <= 16, "lap_delta"].quantile([0.005, 0.995])   # cutoffs from train+valid only
pace = pace[pace.lap_delta.between(lo, hi)].copy()

print(f"rebuilt: pace {pace.shape} | pit {pit.shape} | cutoffs {lo:.4f} to {hi:.4f}")

# ---------- verification against the stored tables ----------
if VERIFY:
    for nm, new, path in [("pace", pace, "model_pace.parquet"), ("pit", pit, "model_pit.parquet")]:
        old = pd.read_parquet(find(path))
        k = ["Round","Driver","LapNumber"]
        m = new.merge(old, on=k, suffixes=("_n","_o"))
        cols, bad = [c for c in old.columns if c not in k], []
        for c in cols:
            a, b = m[c+"_n"], m[c+"_o"]
            try: ok = np.allclose(pd.to_numeric(a, errors="coerce"), pd.to_numeric(b, errors="coerce"), equal_nan=True)
            except Exception: ok = (a.astype(str) == b.astype(str)).all()
            if not ok: bad.append(c)
        print(f"  {nm}: rows {len(new)} vs {len(old)} | {len(cols)-len(bad)}/{len(cols)} columns identical"
              + (f" | differs: {bad}" if bad else ""))

if OVERWRITE:
    pace.to_parquet("model_pace.parquet", index=False)
    pit.to_parquet("model_pit.parquet", index=False)
    print("stored tables overwritten")
else:
    print("stored tables left untouched (set OVERWRITE = True to replace them)")


rebuilt: pace (17758, 71) | pit (22891, 70) | cutoffs -2.7492 to 8.6111
  pace: rows 17758 vs 17758 | 68/68 columns identical
  pit: rows 22891 vs 22891 | 67/67 columns identical
stored tables left untouched (set OVERWRITE = True to replace them)


## Section 4. Does historical data help?

2026 changed the cars, so 2022-2025 lap times are not directly comparable. These three
cells test what, if anything, transfers across the regulation change. Two of the three
tests come out negative and are reported that way.

### Cell 10. Driver-at-circuit affinity (test 9B)
Does a driver's past record at *this* circuit predict his 2026 result there?

In [ ]:
# ============================================================
# CELL 9B - HISTORICAL FEATURES + THE TEST THAT DECIDES
# Builds 3 columns from 2022-2025:
#   driver_circuit_affinity : driver's gap to his OWN team-mate at that
#                             circuit (car-free, so the 2026 reset is fine)
#   circuit_shuffle         : average places moved from grid to finish
#   circuit_dnf_rate        : retirements per entry at that circuit
# Then tests whether the affinity actually predicts 2026. Keep only if it does.
# ============================================================

# finds each file in the current folder, Grp assignment, or Grp assignment\Data
BASE = os.path.join(os.path.expanduser("~"), "OneDrive", "Desktop", "Yug", "Sem 5", "ML", "Grp assignment")
def find(name):
    for d in [os.getcwd(), BASE, os.path.join(BASE, "Data")]:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"{name} not found in {os.getcwd()}, {BASE} or {os.path.join(BASE, 'Data')}")
def P(name): return os.path.join(BASE, name)           # where outputs are saved
hist = pd.read_parquet(find("hist_results.parquet"))
hist["Position"] = pd.to_numeric(hist["Position"], errors="coerce")
hist["GridPosition"] = pd.to_numeric(hist["GridPosition"], errors="coerce")

def vs_teammate(df, col):
    """driver's value minus his team-mate's, same session. Strips the car out."""
    mate = df.groupby(["Year","Round","TeamName"])[col].transform(
        lambda s: (s.sum() - s) / (len(s) - 1) if len(s) > 1 else np.nan)
    return df[col] - mate

# ---------- 1. driver skill at each circuit, from qualifying ----------
hq = hist[(hist.Session == "Q") & hist.Position.notna()].copy()
hq["vs_mate"] = vs_teammate(hq, "Position")
hq = hq.dropna(subset=["vs_mate"])
affinity = (hq.groupby(["Abbreviation","Location"])
              .agg(aff=("vs_mate","mean"), aff_n=("vs_mate","size")).reset_index()
              .rename(columns={"Abbreviation":"Driver"}))
driver_overall = hq.groupby("Abbreviation").vs_mate.mean().rename("aff_overall")
affinity = affinity.merge(driver_overall, left_on="Driver", right_index=True, how="left")
affinity["driver_circuit_affinity"] = affinity.aff - affinity.aff_overall   # better/worse than his own norm

# ---------- 2. circuit behaviour, from races ----------
hr = hist[hist.Session == "R"].copy()
hr["dnf"] = hr.Status.astype(str).str.contains("Retired|Accident|Collision|Engine|Gearbox", case=False, na=False).astype(int)
fin = hr[(hr.dnf == 0) & hr.Position.notna() & hr.GridPosition.notna() & (hr.GridPosition > 0)]
circuit = pd.DataFrame({
    "circuit_shuffle":  fin.assign(m=(fin.Position - fin.GridPosition).abs()).groupby("Location").m.mean(),
    "circuit_dnf_rate": hr.groupby("Location").dnf.mean(),
    "circuit_races":    hr.groupby("Location").Round.count() / 20,
}).reset_index()
print(f"history: {hist.Year.nunique()} seasons, {hist.Location.nunique()} circuits, "
      f"{len(affinity)} driver-circuit pairs\n")

# ============================================================
# THE TEST: does 2022-25 affinity predict 2026 at the SAME circuit?
# ============================================================
qres  = pd.read_excel(find("F1_2026_qualifying.xlsx"), sheet_name="Quali_Results")
qlaps = pd.read_excel(find("F1_2026_qualifying.xlsx"), sheet_name="Quali_Laps")
loc   = pd.read_excel(find("F1_2026_extra.xlsx"), sheet_name="Event_Info")[["Round","Location"]]
q26 = qres[qres["Q1_sec"].notna()].rename(columns={"Abbreviation":"Driver","TeamName":"Team"})[["Round","Driver","Team","Q1_sec"]].copy()
q26["q1_gap_pct"] = 100 * (q26["Q1_sec"] / q26.groupby("Round")["Q1_sec"].transform("min") - 1)
wet = qlaps.groupby("Round")["Rainfall"].apply(lambda r: int(r.mean() > 0.2)).rename("wet_session").reset_index()
q26 = q26.merge(wet, on="Round", how="left").merge(loc, on="Round", how="left")
q26 = q26[q26.wet_session == 0]                                   # the one wet session is not comparable
q26["vs_mate"] = vs_teammate(q26.assign(Year=2026, TeamName=q26.Team), "q1_gap_pct")
q26 = q26.dropna(subset=["vs_mate"])
q26["vs_mate_resid"] = q26.vs_mate - q26.groupby("Driver").vs_mate.transform("mean")

chk = q26.merge(affinity[["Driver","Location","driver_circuit_affinity","aff_n"]],
                on=["Driver","Location"], how="inner")
chk = chk[chk.aff_n >= 2]                                         # at least 2 past visits
print("=" * 64); print("TEST: 2022-25 driver-at-circuit skill vs 2026 at the same circuit"); print("=" * 64)
if len(chk) < 30:
    print(f"Only {len(chk)} matched rows. Too few to judge: do not use the affinity column.")
else:
    r, p = pearsonr(chk.driver_circuit_affinity, chk.vs_mate_resid)
    rho = spearmanr(chk.driver_circuit_affinity, chk.vs_mate_resid).correlation
    print(f"matched rows: {len(chk)} ({chk.Driver.nunique()} drivers, {chk.Location.nunique()} circuits)")
    print(f"correlation: r = {r:+.3f}  (p = {p:.3f})   rank corr = {rho:+.3f}")
    print()
    if p < 0.05 and r > 0.15:
        print("VERDICT: real signal. Keep driver_circuit_affinity as a feature.")
    elif p < 0.05 and r < -0.15:
        print("VERDICT: significant but BACKWARDS. Do not use it; report it as a finding.")
    else:
        print("VERDICT: no usable signal. Drop driver_circuit_affinity.")
        print("Circuit-level columns (shuffle, dnf rate) are separate and still worth keeping.")

# ---------- 3. save, including the 3 upcoming circuits ----------
UPCOMING = {21: "Las Vegas", 22: "Lusail", 23: "Yas Island"}       # check these match hist Location names
print("\nupcoming circuits found in history:",
      {r: (l in set(circuit.Location)) for r, l in UPCOMING.items()})
print("\ncircuit table (top and bottom by shuffle):")
c = circuit.sort_values("circuit_shuffle")
print(pd.concat([c.head(4), c.tail(4)]).round(2).to_string(index=False))
affinity.to_parquet(P("hist_driver_affinity.parquet"), index=False)
circuit.to_parquet(P("hist_circuit.parquet"), index=False)
print("\nSaved: hist_driver_affinity.parquet, hist_circuit.parquet")

history: 4 seasons, 26 circuits, 729 driver-circuit pairs

TEST: 2022-25 driver-at-circuit skill vs 2026 at the same circuit
matched rows: 229 (19 drivers, 15 circuits)
correlation: r = +0.037  (p = 0.579)   rank corr = +0.032

VERDICT: no usable signal. Drop driver_circuit_affinity.
Circuit-level columns (shuffle, dnf rate) are separate and still worth keeping.

upcoming circuits found in history: {21: True, 22: True, 23: True}

circuit table (top and bottom by shuffle):
     Location  circuit_shuffle  circuit_dnf_rate  circuit_races
       Suzuka             1.62              0.11           4.00
       Monaco             1.86              0.11           4.00
Miami Gardens             2.31              0.20           1.00
       Jeddah             2.47              0.11           4.00
       Austin             3.82              0.09           4.00
    São Paulo             3.83              0.16           4.00
    Melbourne             3.93              0.23           3.95
    Las Veg

### Cell 11. Fairness checks on the circuit tests (test 9C)
Re-runs the test with a matching metric, and tries circuit-type grouping instead of circuit identity.

In [ ]:
# ============================================================
# CELL 9C - TWO FAIRNESS CHECKS BEFORE WE DECIDE ON HISTORY
# Check 1: re-run the driver-at-circuit test with a MATCHING metric.
#          (9B compared 2022-25 final qualifying position against 2026 Q1 gap.
#           Different measures can hide a real effect.)
# Check 2: does a circuit's 2022-25 behaviour predict its 2026 behaviour?
#          (grid-to-finish shuffle, and retirement rate)
# ============================================================

BASE = os.path.join(os.path.expanduser("~"), "OneDrive", "Desktop", "Yug", "Sem 5", "ML", "Grp assignment")
def find(name):
    for d in [os.getcwd(), BASE, os.path.join(BASE, "Data")]:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"{name} not found in {os.getcwd()}, {BASE} or {os.path.join(BASE, 'Data')}")

aff     = pd.read_parquet(find("hist_driver_affinity.parquet"))
circuit = pd.read_parquet(find("hist_circuit.parquet"))
race    = pd.read_parquet(find("model_race.parquet"))
loc     = pd.read_excel(find("F1_2026_extra.xlsx"), sheet_name="Event_Info")[["Round","Location"]]
qres    = pd.read_excel(find("F1_2026_qualifying.xlsx"), sheet_name="Quali_Results")

def ci95(r, n):
    z, se = np.arctanh(r), 1 / np.sqrt(n - 3)
    return np.tanh(z - 1.96 * se), np.tanh(z + 1.96 * se)

def verdict(r, p, name):
    if p < 0.10 and r > 0.30:   print(f"VERDICT: real signal. Keep {name}.")
    elif p < 0.10 and r < -0.30: print(f"VERDICT: significant but BACKWARDS. Do not use {name}.")
    else:                        print(f"VERDICT: no usable signal. Drop {name}.")

# ---------- CHECK 1: driver-at-circuit, matching metric ----------
print("=" * 66); print("CHECK 1: driver-at-circuit skill, same metric on both sides"); print("=" * 66)
q = qres.rename(columns={"Abbreviation":"Driver","TeamName":"Team"}).copy()
q["Position"] = pd.to_numeric(q["Position"], errors="coerce")
q = q[q.Position.notna() & (q.Round != 20)].drop(columns=["Location"], errors="ignore").merge(loc, on="Round", how="left")   # round 20 was wet
mate = q.groupby(["Round","Team"]).Position.transform(lambda s: (s.sum() - s) / (len(s) - 1) if len(s) > 1 else np.nan)
q["vs_mate"] = q.Position - mate
q = q.dropna(subset=["vs_mate"])
q["vs_mate_resid"] = q.vs_mate - q.groupby("Driver").vs_mate.transform("mean")
chk = q.merge(aff[["Driver","Location","driver_circuit_affinity","aff_n"]], on=["Driver","Location"])
chk = chk[chk.aff_n >= 2]
r, p = pearsonr(chk.driver_circuit_affinity, chk.vs_mate_resid); lo, hi = ci95(r, len(chk))
print(f"matched rows: {len(chk)} | r = {r:+.3f}  (p = {p:.3f}) | 95% range for r: {lo:+.2f} to {hi:+.2f}")
verdict(r, p, "driver_circuit_affinity")

# ---------- CHECK 2: circuit behaviour, 2022-25 vs 2026 ----------
print("\n" + "=" * 66); print("CHECK 2: does a circuit's history predict its 2026 race?"); print("=" * 66)
r26 = race[race.Round != 20].drop(columns=["Location"], errors="ignore").merge(loc, on="Round", how="left")   # round 20 was wet
fin = r26[(r26.dnf == 0) & (r26.GridPosition > 0)]
now = pd.DataFrame({"shuffle_2026": fin.assign(m=(fin.finish_pos - fin.GridPosition).abs()).groupby("Location").m.mean(),
                    "dnf_2026":     r26.groupby("Location").dnf.mean()}).reset_index()
cc = now.merge(circuit[circuit.circuit_races >= 2], on="Location")
print(f"circuits matched: {len(cc)} (need 2+ seasons of history)\n")
for hist_col, now_col, nm in [("circuit_shuffle","shuffle_2026","circuit_shuffle"), ("circuit_dnf_rate","dnf_2026","circuit_dnf_rate")]:
    r, p = pearsonr(cc[hist_col], cc[now_col]); rho = spearmanr(cc[hist_col], cc[now_col]).correlation
    print(f"{nm:18s} r = {r:+.3f}  (p = {p:.3f})  rank corr = {rho:+.3f}")
    verdict(r, p, nm); print()
print("circuit by circuit (history vs 2026 shuffle):")
print(cc[["Location","circuit_shuffle","shuffle_2026"]].sort_values("circuit_shuffle").round(2).to_string(index=False))

# ---------- what history says about the three unraced circuits ----------
UPCOMING = {"Las Vegas": 21, "Lusail": 22, "Yas Island": 23}
print("\nthe three unraced circuits, from 2022-25 history:")
print(circuit[circuit.Location.isin(UPCOMING)][["Location","circuit_shuffle","circuit_dnf_rate","circuit_races"]].round(2).to_string(index=False))
print(f"(all-circuit average shuffle: {circuit.circuit_shuffle.mean():.2f} places | retirement rate: {circuit.circuit_dnf_rate.mean():.2f})")

NameError: name 'os' is not defined

### Cell 12. Driver rating from teammate comparison (test 9D)
The test that decides. A driver's qualifying gap to his own teammate cancels out the car, so it survives the regulation change. This is the only historical feature kept.

In [ ]:
# ============================================================
# CELL 9D - DOES HISTORY IMPROVE REAL PREDICTIONS?  (the test that matters)
# 1. Builds a DRIVER RATING from 2022-25: how far ahead of his own team-mate
#    he qualified (car-free, so the 2026 rule change does not break it).
# 2. Checks it against 2026 driver-vs-teammate gaps.
# 3. Puts it inside a Plackett-Luce win/podium model and measures, race by
#    race on rounds 8-20 (each trained only on earlier races), whether
#    predictions get better WITH the history than WITHOUT it.
# Rule fixed in advance: keep history only if the average log P(winner)
# improves by 0.03 or more in BOTH the pre-weekend and post-qualifying cases.
# ============================================================

BASE = os.path.join(os.path.expanduser("~"), "OneDrive", "Desktop", "Yug", "Sem 5", "ML", "Grp assignment")
def find(name):
    for d in [os.getcwd(), BASE, os.path.join(BASE, "Data")]:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"{name} not found in {os.getcwd()}, {BASE} or {os.path.join(BASE, 'Data')}")

hist = pd.read_parquet(find("hist_results.parquet"))
race = pd.read_parquet(find("model_race.parquet"))
qres = pd.read_excel(find("F1_2026_qualifying.xlsx"), sheet_name="Quali_Results")

# ---------- 1. driver rating from 2022-25 ----------
hq = hist[hist.Session == "Q"].copy()
hq["Position"] = pd.to_numeric(hq["Position"], errors="coerce"); hq = hq[hq.Position.notna()]
mate = hq.groupby(["Year","Round","TeamName"]).Position.transform(lambda s: (s.sum() - s) / (len(s) - 1) if len(s) > 1 else np.nan)
hq["vs_mate"] = hq.Position - mate; hq = hq.dropna(subset=["vs_mate"])
d = hq.groupby("Abbreviation").vs_mate.agg(raw="mean", n="size")
d["drv_prior"] = -d.raw * d.n / (d.n + 10)          # positive = stronger driver; shrunk when few sessions
print(f"driver ratings built for {len(d)} drivers (rookies with no history get 0)\n")

# ---------- 2. does the rating predict 2026? ----------
q = qres.rename(columns={"Abbreviation":"Driver","TeamName":"Team"}).copy()
q["Position"] = pd.to_numeric(q["Position"], errors="coerce"); q = q[q.Position.notna() & (q.Round != 20)]
m26 = q.groupby(["Round","Team"]).Position.transform(lambda s: (s.sum() - s) / (len(s) - 1) if len(s) > 1 else np.nan)
q["vs_mate"] = q.Position - m26
now = -q.dropna(subset=["vs_mate"]).groupby("Driver").vs_mate.mean().rename("now_2026")     # positive = stronger
chk = d[d.n >= 20].join(now, how="inner")
r, p = pearsonr(chk.drv_prior, chk.now_2026); z, se = np.arctanh(r), 1 / np.sqrt(len(chk) - 3)
print("=" * 66); print("CHECK: 2022-25 driver rating vs 2026 driver-vs-teammate gap"); print("=" * 66)
print(f"drivers: {len(chk)} | r = {r:+.3f} (p = {p:.3f}) | 95% range: {np.tanh(z-1.96*se):+.2f} to {np.tanh(z+1.96*se):+.2f}\n")

# ---------- 3. Plackett-Luce, with and without history ----------
race = race.sort_values(["Driver","Round"]).copy()
race["d_ewm"] = race.groupby("Driver").finish_pos.transform(lambda s: s.shift(1).ewm(halflife=3).mean())
tm = race.groupby(["Team","Round"]).finish_pos.mean().reset_index().sort_values(["Team","Round"])
tm["t_ewm"] = tm.groupby("Team").finish_pos.transform(lambda s: s.shift(1).ewm(halflife=3).mean())
race = race.merge(tm[["Team","Round","t_ewm"]], on=["Team","Round"], how="left")
for c in ["d_ewm","t_ewm"]:
    race[c] = race[c].fillna(race.groupby("Round")[c].transform("median")).fillna(10)
race["pts_share"] = race.driver_points_sofar / race.groupby("Round").driver_points_sofar.transform("max").replace(0, 1)
race["team_pts_share"] = race.team_points_sofar / race.groupby("Round").team_points_sofar.transform("max").replace(0, 1)
race["log_grid"] = np.log(race.GridPosition.clip(lower=1))
race["drv_prior"] = race.Driver.map(d.drv_prior).fillna(0.0)

def fit_pl(tr, F, lam=1.0, top=3):
    mu, sd = tr[F].mean(), tr[F].std().replace(0, 1)
    races = []
    for _, g in tr.groupby("Round"):
        X, order = ((g[F] - mu) / sd).values, g.finish_pos.values
        races.append((X, [int(np.where(order == k)[0][0]) for k in range(1, top + 1) if (order == k).any()]))
    def nll(w):
        tot = 0.0
        for X, idx in races:
            s, used = X @ w, []
            for i in idx:
                mask = np.ones(len(s), bool); mask[used] = False
                m = s[mask].max(); tot -= s[i] - (m + np.log(np.exp(s[mask] - m).sum())); used.append(i)
        return tot + lam * np.sum(w ** 2)
    return minimize(nll, np.zeros(len(F)), method="L-BFGS-B").x, mu, sd

def simulate(s, n=4000, seed=0):
    rng = np.random.default_rng(seed); order = np.argsort(-(s + rng.gumbel(size=(n, len(s)))), axis=1)
    pos = np.empty_like(order)
    for k in range(len(s)): pos[np.arange(n), order[:, k]] = k + 1
    return (pos == 1).mean(0), (pos <= 3).mean(0)

def walk(F, rounds=range(8, 21)):
    W, P, LL, BR = [], [], [], []
    for r in rounds:
        tr, te = race[(race.Round >= 2) & (race.Round < r)], race[race.Round == r].copy()
        w, mu, sd = fit_pl(tr, F); te["pw"], te["pp"] = simulate(((te[F] - mu) / sd).values @ w)
        win = (te.finish_pos == 1).values
        W.append(int(te.loc[te.pw.idxmax(), "finish_pos"] == 1)); P.append(int((te.nlargest(3, "pp").finish_pos <= 3).sum()))
        LL.append(np.log(max(te.loc[win, "pw"].values[0] if win.any() else 1e-3, 1e-3)))
        BR.append(np.mean((te.pp.values - (te.finish_pos <= 3).values) ** 2))
    return dict(winner=f"{sum(W)}/{len(W)}", podium=np.mean(P), logP=np.mean(LL), brier=np.mean(BR))

PRE  = ["t_ewm", "d_ewm", "pts_share", "team_pts_share"]
POST = ["log_grid", "q1_gap_pct"] + PRE
res = {}
for scen, base in [("pre-weekend", PRE), ("post-qualifying", POST)]:
    res[(scen, "without history")] = walk(base)
    res[(scen, "with history")]    = walk(base + ["drv_prior"])
out = pd.DataFrame(res).T.rename(columns={"winner":"winner picked","podium":"podium hits (of 3)","logP":"avg log P(winner)","brier":"podium Brier"})
print("=" * 66); print("WALK-FORWARD, 13 races (rounds 8-20), each trained on earlier races only"); print("=" * 66)
print(out.round(3).to_string())
print(f"\n(guessing uniformly would score log P(winner) = {np.log(1/22):.2f}; higher is better)")

gain = {s: res[(s, "with history")]["logP"] - res[(s, "without history")]["logP"] for s in ["pre-weekend", "post-qualifying"]}
print(f"\nchange in avg log P(winner) from adding history: pre-weekend {gain['pre-weekend']:+.3f} | post-qualifying {gain['post-qualifying']:+.3f}")
if all(v >= 0.03 for v in gain.values()):
    print("VERDICT: history improves real predictions in both cases. Keep the driver rating.")
elif any(v >= 0.03 for v in gain.values()):
    print("VERDICT: helps in one case only. Use it only there, and say so in the report.")
else:
    print("VERDICT: history does not improve predictions here. Report it as tested, not used.")

driver ratings built for 31 drivers (rookies with no history get 0)

CHECK: 2022-25 driver rating vs 2026 driver-vs-teammate gap
drivers: 22 | r = +0.430 (p = 0.046) | 95% range: +0.01 to +0.72

WALK-FORWARD, 13 races (rounds 8-20), each trained on earlier races only
                                winner picked podium hits (of 3) avg log P(winner) podium Brier
pre-weekend     without history          4/13           1.384615         -2.119019     0.085247
                with history             5/13           1.615385         -1.978259     0.080289
post-qualifying without history         10/13           1.846154         -1.368267     0.068183
                with history            10/13           1.923077         -1.331776     0.068615

(guessing uniformly would score log P(winner) = -3.09; higher is better)

change in avg log P(winner) from adding history: pre-weekend +0.141 | post-qualifying +0.036
VERDICT: history improves real predictions in both cases. Keep the driver rating.


## Section 5. In-race models (1, 2, 3 and 8)

These four models use information from a race already in progress, so they cannot
forecast a future race. Each is compared against a simple baseline.

### Cell 13. Models 1 and 2, lap pace and tyre degradation
One shared fit. Model 1 predicts a lap's pace against the driver's own median lap. Model 2 reads the same fit as a tyre-age curve by compound.

In [ ]:
# ============================================================
# CELL 2 - MODELS 1 AND 2 (REGRESSION, one shared table)
# Model 1: how much faster/slower than his own race pace is a lap?
# Model 2: how fast does each real tyre compound (C1-C5) lose pace?
# Unit   : one clean, dry, green-flag racing lap
# Target : lap_delta (sec) = lap time minus driver's median lap in that race
# ============================================================
pace = pd.read_parquet("model_pace.parquet")
pace = pace[pace["Compound"] != "INTERMEDIATE"].copy()                 # dry running only
for c in ["Compound","Team","circuit_type"]:
    pace[c] = pace[c].astype("category")
train, valid, test = split(pace)
final_train = pd.concat([train, valid])
print(f"laps: train {len(train)} | valid {len(valid)} | test {len(test)}\n")

FEATURES = ["TyreLife","laps_left","compound_C","Compound","Team","circuit_type",
            "gap_ahead","dirty_air","team_fp_deg","team_longrun_gap_pct"]

# ---------- MODEL B: linear, separate tyre-wear slope for every C-compound ----------
class LinearDeg:
    def prep(self, X):
        Z = X[["TyreLife","laps_left","dirty_air"]].astype(float).copy()
        for c in [1, 2, 4, 5]:                                         # C3 is the reference compound
            d = (X["compound_C"] == c).astype(float)
            Z[f"C{c}"] = d
            Z[f"C{c}_x_age"] = d * X["TyreLife"].astype(float)
        return Z
    def fit(self, X, y): self.m = LinearRegression().fit(self.prep(X), y); return self
    def predict(self, X): return self.m.predict(self.prep(X))

# ---------- MODEL C: gradient boosting ----------
def Boosting():
    return HistGradientBoostingRegressor(categorical_features="from_dtype", max_iter=150, learning_rate=0.05,
            max_depth=3, min_samples_leaf=100, early_stopping=False, random_state=0)   # FIX 3: no hidden random split

models = {"A. Baseline (average)": Baseline, "B. Linear (C-compound x tyre age)": LinearDeg, "C. Gradient boosting": Boosting}
def score(y, p): return [np.sqrt(mean_squared_error(y, p)), mean_absolute_error(y, p), r2_score(y, p)]

print("=" * 60); print("MODEL 1: LAP PACE"); print("=" * 60)
rows, fitted, val = [], {}, {}
for name, make in models.items():
    pv = make().fit(train[FEATURES], train.lap_delta.values).predict(valid[FEATURES])
    val[name] = mean_absolute_error(valid.lap_delta, pv)
    fitted[name] = make().fit(final_train[FEATURES], final_train.lap_delta.values)
    pt = fitted[name].predict(test[FEATURES])
    rows.append([name, *score(valid.lap_delta, pv), *score(test.lap_delta, pt)])
m1_table = pd.DataFrame(rows, columns=["Model","Val RMSE","Val MAE","Val R2","Test RMSE","Test MAE","Test R2"]).round(3)
print(m1_table.to_string(index=False))

# ---------- the leakage check (headline result for the report) ----------
WX = FEATURES + ["AirTemp","TrackTemp","Humidity","WindSpeed"]
leaky = lambda: HistGradientBoostingRegressor(categorical_features="from_dtype", max_iter=300, learning_rate=0.05,
                                              max_depth=4, early_stopping=False, random_state=0)
p = np.zeros(len(final_train))
for tr, te in KFold(5, shuffle=True, random_state=0).split(final_train):
    p[te] = leaky().fit(final_train[WX].iloc[tr], final_train.lap_delta.values[tr]).predict(final_train[WX].iloc[te])
leak = {"random split R2": r2_score(final_train.lap_delta, p),
        "time split R2":   r2_score(test.lap_delta, leaky().fit(final_train[WX], final_train.lap_delta.values).predict(test[WX]))}
print(f"\nLeakage check (boosting WITH weather): random split R2 = {leak['random split R2']:.3f} | "
      f"time split R2 = {leak['time split R2']:.3f}")
print("Same model, same data. Only the split changed. The gap is memorised races, not learned pace.")

# ============================================================
# MODEL 2: TYRE DEGRADATION BY REAL COMPOUND (C1 hardest ... C5 softest)
# Read inside each stint, so driver, car and race are held fixed.
# ============================================================
print("\n" + "=" * 60); print("MODEL 2: TYRE DEGRADATION BY C-COMPOUND"); print("=" * 60)
lin = fitted["B. Linear (C-compound x tyre age)"]
coef = dict(zip(lin.prep(final_train[FEATURES]).columns, lin.m.coef_))
fuel = coef["laps_left"]                     # sec per lap of fuel on board (car gets this much faster per lap)
print(f"Fuel + track-evolution effect: {fuel:.3f} sec per lap")
srows = []
for _, s in final_train.groupby(["Round","Driver","Stint"]):
    if len(s) >= 8 and s["TyreLife"].nunique() > 2:
        raw = np.polyfit(s["TyreLife"].astype(float), s["LapTime_sec"], 1)[0]
        srows.append({"compound_C": int(s["compound_C"].iloc[0]), "label": str(s["Compound"].iloc[0]),
                      "raw_slope": raw, "wear_slope": raw + fuel})
deg = (pd.DataFrame(srows).groupby("compound_C")
       .agg(stints=("raw_slope","size"), raw_sec_per_lap=("raw_slope","median"),
            wear_sec_per_lap=("wear_slope","median"),
            labelled_as=("label", lambda s: "/".join(sorted(s.unique())))).round(3))
deg["wear_over_20_laps"] = (deg["wear_sec_per_lap"] * 20).round(2)
print(deg.to_string())
print("wear_sec_per_lap = within-stint slope + fuel effect. An association, not a causal wear rate.")

# curve for the report figure: linear model, typical race state
curve = pd.DataFrame([{"compound_C": c, "TyreLife": a} for c in [1,2,3,4,5] for a in range(1, 36)])
curve["laps_left"] = final_train["laps_left"].median(); curve["dirty_air"] = 0
curve["pred_lap_delta"] = lin.predict(curve)
curve.to_csv("model2_degradation_curve.csv", index=False)

# ---------- save the validation winner ----------
best = min(val, key=val.get)
print(f"\nPicked on VALIDATION, not test -> {best}")
joblib.dump(fitted[best] if not best.startswith("B.") else lin.m, "saved_models/model1_pace.pkl")
json.dump({"features": FEATURES, "target": "lap_delta", "model": best, "train_rounds": "1-16", "test_rounds": "17-20",
           "degradation_sec_per_lap_by_C": deg["wear_sec_per_lap"].to_dict(), "leakage_check": leak},
          open("saved_models/model1_pace.json", "w"), indent=2, default=float)
m1_table.to_csv("results_model1.csv", index=False); deg.to_csv("results_model2.csv")
print("Saved: saved_models/model1_pace.pkl, results_model1.csv, results_model2.csv, model2_degradation_curve.csv")

laps: train 11320 | valid 3028 | test 3297

MODEL 1: LAP PACE
                            Model  Val RMSE  Val MAE  Val R2  Test RMSE  Test MAE  Test R2
            A. Baseline (average)     1.224    0.850  -0.000      1.196     0.869   -0.027
B. Linear (C-compound x tyre age)     1.057    0.727   0.255      1.044     0.733    0.218
             C. Gradient boosting     1.045    0.722   0.271      1.060     0.754    0.193

Leakage check (boosting WITH weather): random split R2 = 0.570 | time split R2 = -0.164
Same model, same data. Only the split changed. The gap is memorised races, not learned pace.

MODEL 2: TYRE DEGRADATION BY C-COMPOUND
Fuel + track-evolution effect: 0.034 sec per lap
            stints  raw_sec_per_lap  wear_sec_per_lap       labelled_as  wear_over_20_laps
compound_C                                                                                
1               40           -0.016             0.018              HARD               0.36
2              187           

### Cell 14. Model 3, will this driver pit next lap?
Only 3.76% of laps are pit laps, so this is scored with PR-AUC rather than accuracy.

In [ ]:
# ============================================================
# CELL 3 - MODEL 3 (CLASSIFICATION)
# Question: will this driver pit at the end of the NEXT lap?
# Target  : pit_next_lap (1 = yes). About 3.8% of laps.
# Every feature is known at the end of the current lap.
# ============================================================
pit = pd.read_parquet("model_pit.parquet")
for c in ["Compound","circuit_type"]:
    pit[c] = pit[c].astype("category")

NUM = ["TyreLife","Stint","LapNumber","laps_left","frac_done","Position","lap_delta_prev","pace_trend_3",
       "sc_or_vsc","yellow",                                   # race state
       "gap_ahead","gap_behind","cars_pitted_prev2",           # strategist features: rivals + pit waves
       "compound_C","pit_lane_median_sec","team_fp_deg"]       # tyre + track
CAT = ["Compound","circuit_type"]
FEATURES = NUM + CAT
train, valid, test = split(pit)
final_train = pd.concat([train, valid])
print(f"train R1-12 {len(train)} | valid R13-16 {len(valid)} | test R17-20 {len(test)}")
print(f"pit rate: train {train.pit_next_lap.mean()*100:.2f}% | test {test.pit_next_lap.mean()*100:.2f}%\n")

def make_logistic():
    pre = ColumnTransformer([("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM),
                             ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])
    return Pipeline([("prep", pre), ("clf", LogisticRegression(max_iter=3000, class_weight="balanced"))])
def make_boosting():
    return HistGradientBoostingClassifier(categorical_features="from_dtype", max_iter=250, learning_rate=0.05,
            max_depth=4, min_samples_leaf=50, early_stopping=False, random_state=0)

def best_threshold(p, y):                       # chosen on VALIDATION only
    ts = np.arange(0.02, 0.81, 0.01)
    return float(ts[np.argmax([f1_score(y, (p >= t).astype(int), zero_division=0) for t in ts])])

models = {"B. Logistic regression": make_logistic, "C. Gradient boosting": make_boosting}
rows, fitted, thr, val_pr = [], {}, {}, {}
y_te = test.pit_next_lap.values
rows.append(dict(Model="A. Baseline (never pit)", Threshold=0.5, Val_PR_AUC=train.pit_next_lap.mean(),
                 PR_AUC=average_precision_score(y_te, np.full(len(y_te), train.pit_next_lap.mean())),
                 ROC_AUC=0.5, Precision=0.0, Recall=0.0, F1=0.0))
for name, make in models.items():
    pv = make().fit(train[FEATURES], train.pit_next_lap).predict_proba(valid[FEATURES])[:, 1]
    thr[name] = best_threshold(pv, valid.pit_next_lap.values)
    val_pr[name] = average_precision_score(valid.pit_next_lap, pv)
    fitted[name] = make().fit(final_train[FEATURES], final_train.pit_next_lap)        # FIX 4: refit on 1-16
    p = fitted[name].predict_proba(test[FEATURES])[:, 1]; hard = (p >= thr[name]).astype(int)
    rows.append(dict(Model=name, Threshold=thr[name], Val_PR_AUC=val_pr[name],
                     PR_AUC=average_precision_score(y_te, p), ROC_AUC=roc_auc_score(y_te, p),
                     Precision=precision_score(y_te, hard, zero_division=0),
                     Recall=recall_score(y_te, hard, zero_division=0), F1=f1_score(y_te, hard, zero_division=0)))
m3_table = pd.DataFrame(rows).round(3)
print(m3_table.to_string(index=False))

best = max(val_pr, key=val_pr.get)
p_best = fitted[best].predict_proba(test[FEATURES])[:, 1]
print(f"\nPicked on VALIDATION PR-AUC -> {best} (threshold {thr[best]:.2f})")
print("Confusion matrix on test (rows = actual, cols = predicted)")
print(pd.DataFrame(confusion_matrix(y_te, (p_best >= thr[best]).astype(int)),
                   index=["no pit", "pit"], columns=["pred no pit", "pred pit"]))
print(f"\nPR-AUC lift over baseline: {m3_table.loc[m3_table.Model == best, 'PR_AUC'].item() / m3_table.PR_AUC.iloc[0]:.1f}x")

joblib.dump(fitted[best], "saved_models/model3_pit.pkl")
json.dump({"features": FEATURES, "numeric": NUM, "categorical": CAT, "threshold": thr[best], "model": best,
           "target": "pit_next_lap", "train_rounds": "1-16", "test_rounds": "17-20"},
          open("saved_models/model3_pit.json", "w"), indent=2)
m3_table.to_csv("results_model3.csv", index=False)
print("Saved: saved_models/model3_pit.pkl, results_model3.csv")

train R1-12 13838 | valid R13-16 4198 | test R17-20 4855
pit rate: train 3.72% | test 3.63%

                  Model  Threshold  Val_PR_AUC  PR_AUC  ROC_AUC  Precision  Recall    F1
A. Baseline (never pit)       0.50       0.037   0.036    0.500      0.000   0.000 0.000
 B. Logistic regression       0.72       0.088   0.117    0.742      0.096   0.534 0.162
   C. Gradient boosting       0.14       0.071   0.136    0.786      0.128   0.227 0.164

Picked on VALIDATION PR-AUC -> B. Logistic regression (threshold 0.72)
Confusion matrix on test (rows = actual, cols = predicted)
        pred no pit  pred pit
no pit         3789       890
pit              82        94

PR-AUC lift over baseline: 3.3x
Saved: saved_models/model3_pit.pkl, results_model3.csv


### Cell 15. Model 8, stint consistency

In [ ]:
# ============================================================
# CELL 6 - MODEL 8 (REGRESSION)
# Question: how consistent will a driver's lap times be in a stint?
# Unit    : one stint (one driver, one set of tyres), 8+ clean laps
# Target  : consistency_sec = spread of lap times AFTER removing the
#           tyre-wear trend. Lower = more consistent.
# ============================================================

pace = pd.read_parquet("model_pace.parquet")
pace = pace[pace["Compound"] != "INTERMEDIATE"].copy()

rows = []
for (rnd, drv, stint), s in pace.groupby(["Round","Driver","Stint"]):
    if len(s) < 8 or s["TyreLife"].nunique() < 3:
        continue
    x, y = s["TyreLife"].values.astype(float), s["LapTime_sec"].values
    slope, icpt = np.polyfit(x, y, 1)                                   # tyre-wear trend
    rows.append({"Round": rnd, "Driver": drv, "Stint": stint,
                 "consistency_sec": (y - (slope * x + icpt)).std(ddof=1),
                 "Compound": s["Compound"].iloc[0], "compound_C": s["compound_C"].iloc[0], "Team": s["Team"].iloc[0],
                 "circuit_type": s["circuit_type"].iloc[0], "start_tyre_age": s["TyreLife"].min(),
                 "laps_left_at_start": s["laps_left"].max(),             # FIX 1 applied: true race distance
                 "dirty_air_share": s["dirty_air"].mean(),               # NOTE: known only during the stint
                 "team_fp_deg": s["team_fp_deg"].iloc[0]})
st = pd.DataFrame(rows)

dr = st.groupby(["Driver","Round"])["consistency_sec"].mean().reset_index().sort_values(["Driver","Round"])
dr["driver_past_consistency"] = dr.groupby("Driver")["consistency_sec"].transform(lambda s: s.shift(1).expanding().mean())
st = st.merge(dr[["Driver","Round","driver_past_consistency"]], on=["Driver","Round"], how="left")
st["driver_past_consistency"] = st["driver_past_consistency"].fillna(st.loc[st.Round <= 16, "consistency_sec"].median())

NUM = ["start_tyre_age","laps_left_at_start","compound_C","dirty_air_share","team_fp_deg","driver_past_consistency"]
CAT = ["Team","circuit_type"]
FEATURES = NUM + CAT
train, valid, test = split(st)
final_train = pd.concat([train, valid])
print(f"stints: train {len(train)} | valid {len(valid)} | test {len(test)}")
print(f"typical consistency: {st.consistency_sec.median():.3f} sec around the wear trend\n")

def prep(): return ColumnTransformer([("num", StandardScaler(), NUM), ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])
models = {"A. Baseline (average)": lambda: Baseline(),
          "B. Ridge regression":   lambda: Pipeline([("p", prep()), ("m", Ridge(alpha=5.0))]),
          "C. Gradient boosting":  lambda: Pipeline([("p", prep()), ("m", HistGradientBoostingRegressor(
                                       max_iter=150, learning_rate=0.05, max_depth=3, min_samples_leaf=20,
                                       early_stopping=False, random_state=0))])}
def score(y, p): return [np.sqrt(mean_squared_error(y, p)), mean_absolute_error(y, p), r2_score(y, p)]
rows, fitted, val = [], {}, {}
for name, make in models.items():
    pv = make().fit(train[FEATURES], train.consistency_sec).predict(valid[FEATURES])
    val[name] = mean_absolute_error(valid.consistency_sec, pv)
    fitted[name] = make().fit(final_train[FEATURES], final_train.consistency_sec)
    rows.append([name, *score(valid.consistency_sec, pv), *score(test.consistency_sec, fitted[name].predict(test[FEATURES]))])
m8_table = pd.DataFrame(rows, columns=["Model","Val RMSE","Val MAE","Val R2","Test RMSE","Test MAE","Test R2"]).round(3)
print(m8_table.to_string(index=False))

print("\nMedian consistency by real compound (sec, descriptive only):")
print(st.groupby("compound_C")["consistency_sec"].agg(["count","median"]).round(3))
print("\nMedian consistency in traffic vs clean air (sec):")
print(st.groupby(st.dirty_air_share > 0.5)["consistency_sec"].median().rename({False:"mostly clean air", True:"mostly in traffic"}).round(3))

best = min(val, key=val.get)
print(f"\nPicked on VALIDATION, not test -> {best}")
if best.startswith("A."):
    print("Baseline won on validation: no model saved. Model 8 stays off the website.")
else:
    joblib.dump(fitted[best], "saved_models/model8_consistency.pkl")
    json.dump({"features": FEATURES, "numeric": NUM, "categorical": CAT, "target": "consistency_sec", "model": best,
               "train_rounds": "1-16", "test_rounds": "17-20"}, open("saved_models/model8_consistency.json", "w"), indent=2)
    print("Saved: saved_models/model8_consistency.pkl")
m8_table.to_csv("results_model8.csv", index=False)

stints: train 553 | valid 139 | test 146
typical consistency: 0.478 sec around the wear trend

                Model  Val RMSE  Val MAE  Val R2  Test RMSE  Test MAE  Test R2
A. Baseline (average)     0.444    0.284  -0.074      0.395     0.307   -0.006
  B. Ridge regression     0.417    0.303   0.055      0.381     0.281    0.063
 C. Gradient boosting     0.416    0.303   0.059      0.388     0.272    0.027

Median consistency by real compound (sec, descriptive only):
            count  median
compound_C               
1.0            40   0.442
2.0           204   0.478
3.0           197   0.437
4.0           251   0.450
5.0           146   0.686

Median consistency in traffic vs clean air (sec):
dirty_air_share
mostly clean air     0.486
mostly in traffic    0.461
Name: consistency_sec, dtype: float64

Picked on VALIDATION, not test -> A. Baseline (average)
Baseline won on validation: no model saved. Model 8 stays off the website.


## Section 6. Forecast models (5, 6 and 7)

Models 5, 6 and 7 are the forecast models. Each has a v1 and a v2. The v1 cells are
kept because the report uses them as comparison rows, and because Cell 18 reads a
file written by Cell 17.

The v2 models replaced the v1 regressions because a squared-error regression on
finishing position pulls every prediction toward mid-grid. The lowest position the
v1 model ever predicted was 2.28, so it could not predict a winner at all.

### Cell 16. Model 5 v1, Q1 gap (superseded, kept for comparison)
Targets the gap to the fastest Q1 time. Q1 is the wrong target: the grid is decided in Q3, and the fast cars do not push in Q1.

In [ ]:
# ============================================================
# CELL 4 - MODEL 5 (REGRESSION)
# Question: how far off the fastest Q1 time will each driver be?
# Target  : q1_gap_pct (% behind the fastest Q1 lap)
# Uses only what is known before qualifying: past form, this
# weekend's practice pace, the stewards' rain forecast.
# ============================================================
q = pd.read_parquet("model_quali.parquet")
q = q[q["Round"] >= 2].copy()                     # round 1 has no history
FEATURES = ["team_form","team_last3","driver_form","driver_last",          # past form
            "driver_fp_gap_pct","team_fp_gap_pct","team_longrun_gap_pct",  # this weekend's practice
            "quali_rain_risk","TrackTemp","n_corners"]                      # conditions + track
train, valid, test = split(q)
final_train = pd.concat([train, valid])
print(f"train R2-12 {len(train)} | valid R13-16 {len(valid)} | test R17-20 {len(test)}")
print("wet sessions in train+valid:", int(final_train.wet_session.sum()), "| in test:", int(test.wet_session.sum()), "\n")

models = {
 "A. Baseline (average gap)": lambda: Baseline(),
 "B. Ridge regression":       lambda: Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=1.0))]),
 "C. Gradient boosting":      lambda: HistGradientBoostingRegressor(max_iter=150, learning_rate=0.05, max_depth=3,
                                         min_samples_leaf=15, early_stopping=False, random_state=0),
}
def score(y, p): return [np.sqrt(mean_squared_error(y, p)), mean_absolute_error(y, p), r2_score(y, p)]
rows, fitted, val = [], {}, {}
dry = (test.wet_session == 0).values
for name, make in models.items():
    pv = make().fit(train[FEATURES], train.q1_gap_pct).predict(valid[FEATURES])
    val[name] = mean_absolute_error(valid.q1_gap_pct, pv)
    fitted[name] = make().fit(final_train[FEATURES], final_train.q1_gap_pct)
    p = fitted[name].predict(test[FEATURES])
    rows.append([name, *score(valid.q1_gap_pct, pv), *score(test.q1_gap_pct, p),
                 *score(test.q1_gap_pct[dry], p[dry])])
m5_table = pd.DataFrame(rows, columns=["Model","Val RMSE","Val MAE","Val R2","Test RMSE","Test MAE","Test R2",
                                       "Dry RMSE","Dry MAE","Dry R2"]).round(3)
print(m5_table.to_string(index=False))
print("\n(gaps in % of the fastest lap; 1% of an 80 sec lap = 0.8 sec)")

# how much does practice alone tell us? (simple check for the report)
r_fp = np.corrcoef(final_train.driver_fp_gap_pct, final_train.q1_gap_pct)[0, 1]
print(f"Correlation, practice gap vs Q1 gap (rounds 2-16): {r_fp:.2f}")

best = min(val, key=val.get)
print(f"\nPicked on VALIDATION, not test -> {best}")
joblib.dump(fitted[best], "saved_models/model5_quali.pkl")
json.dump({"features": FEATURES, "target": "q1_gap_pct", "model": best, "train_rounds": "2-16", "test_rounds": "17-20"},
          open("saved_models/model5_quali.json", "w"), indent=2)
m5_table.to_csv("results_model5.csv", index=False)
print("Saved: saved_models/model5_quali.pkl, results_model5.csv")

train R2-12 241 | valid R13-16 86 | test R17-20 86
wet sessions in train+valid: 0 | in test: 22 

                    Model  Val RMSE  Val MAE  Val R2  Test RMSE  Test MAE  Test R2  Dry RMSE  Dry MAE  Dry R2
A. Baseline (average gap)     1.047    0.831  -0.011      1.445     0.858   -0.007     0.857    0.715  -0.025
      B. Ridge regression     0.638    0.492   0.625      1.336     0.600    0.139     0.450    0.352   0.717
     C. Gradient boosting     0.584    0.445   0.686      1.347     0.632    0.125     0.514    0.415   0.631

(gaps in % of the fastest lap; 1% of an 80 sec lap = 0.8 sec)
Correlation, practice gap vs Q1 gap (rounds 2-16): 0.82

Picked on VALIDATION, not test -> C. Gradient boosting
Saved: saved_models/model5_quali.pkl, results_model5.csv


### Cell 17. Models 6 and 7 v1, regression (superseded, kept for comparison)
Writes `model6_finish.json`, which Cell 19 reads to build its comparison row.

In [ ]:
# ============================================================
# CELL 5 - MODELS 6 AND 7 (one row = one driver in one race)
# Model 6 (regression)    : where will the driver finish?
# Model 7 (classification): will the driver score points (top 10)?
# Uses only what is known before the race starts.
# ============================================================
r = pd.read_parquet("model_race.parquet")
r = r[r["Round"] >= 2].copy()
FEATURES = ["GridPosition","q1_gap_pct","sprint_pos",                                   # this weekend
            "team_longrun_gap_pct","team_fp_deg","driver_fp_gap_pct",                     # practice
            "driver_avg_finish","driver_points_sofar","driver_dnf_rate",                  # driver history
            "team_points_sofar","team_avg_finish","team_dnf_rate",                         # team history
            "race_rain_risk","pit_lane_median_sec"]                                        # conditions + track
train, valid, test = split(r)
final_train = pd.concat([train, valid])
print(f"train R2-12 {len(train)} | valid R13-16 {len(valid)} | test R17-20 {len(test)}")
print(f"points-finish rate: {r.points_finish.mean()*100:.1f}% | retirements: {r.dnf.mean()*100:.1f}%\n")

def race_spearman(df, pred):
    d = df.assign(p=pred)
    return np.mean([spearmanr(g.finish_pos, g.p).correlation for _, g in d.groupby("Round")])

# ============================================================
# MODEL 6 - FINISHING POSITION
# ============================================================
print("=" * 60); print("MODEL 6: FINISHING POSITION"); print("=" * 60)
m6 = {"A. Baseline (finish = grid)": None,
      "B. Ridge regression":  lambda: Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=1.0))]),
      "C. Gradient boosting": lambda: HistGradientBoostingRegressor(max_iter=150, learning_rate=0.05, max_depth=3,
                                         min_samples_leaf=20, early_stopping=False, random_state=0)}
rows, fit6, val6 = [], {}, {}
for name, make in m6.items():
    if make is None:
        pv, pt = valid.GridPosition.values, test.GridPosition.values
    else:
        pv = make().fit(train[FEATURES], train.finish_pos).predict(valid[FEATURES])
        val6[name] = mean_absolute_error(valid.finish_pos, pv)
        fit6[name] = make().fit(final_train[FEATURES], final_train.finish_pos)
        pt = fit6[name].predict(test[FEATURES])
    fin = (test.dnf == 0).values
    rows.append([name, mean_absolute_error(valid.finish_pos, pv), race_spearman(valid, pv),
                 mean_absolute_error(test.finish_pos, pt), r2_score(test.finish_pos, pt), race_spearman(test, pt),
                 mean_absolute_error(test.finish_pos[fin], pt[fin])])
m6_table = pd.DataFrame(rows, columns=["Model","Val MAE","Val Spearman","Test MAE","Test R2","Test Spearman",
                                       "Test MAE (finishers only)"]).round(3)
print(m6_table.to_string(index=False))
print("MAE in positions. Spearman = how well the predicted ORDER matches the real one (1 = perfect).\n")

# ============================================================
# MODEL 7 - POINTS FINISH
# ============================================================
print("=" * 60); print("MODEL 7: POINTS FINISH"); print("=" * 60)
m7 = {"A. Baseline (grid top 10)": None,
      "B. Logistic regression": lambda: Pipeline([("sc", StandardScaler()), ("m", LogisticRegression(max_iter=3000))]),
      "C. Gradient boosting":   lambda: HistGradientBoostingClassifier(max_iter=150, learning_rate=0.05, max_depth=3,
                                           min_samples_leaf=20, early_stopping=False, random_state=0)}
rows, fit7, val7 = [], {}, {}
y = test.points_finish
for name, make in m7.items():
    if make is None:
        hard = (test.GridPosition <= 10).astype(int).values; auc_p = -test.GridPosition.values
        v_auc = roc_auc_score(valid.points_finish, -valid.GridPosition)
    else:
        pv = make().fit(train[FEATURES], train.points_finish).predict_proba(valid[FEATURES])[:, 1]
        v_auc = val7[name] = roc_auc_score(valid.points_finish, pv)
        fit7[name] = make().fit(final_train[FEATURES], final_train.points_finish)
        auc_p = fit7[name].predict_proba(test[FEATURES])[:, 1]; hard = (auc_p >= 0.5).astype(int)
    rows.append([name, v_auc, accuracy_score(y, hard), precision_score(y, hard), recall_score(y, hard),
                 f1_score(y, hard), roc_auc_score(y, auc_p)])
m7_table = pd.DataFrame(rows, columns=["Model","Val ROC-AUC","Test Accuracy","Precision","Recall","F1","Test ROC-AUC"]).round(3)
print(m7_table.to_string(index=False))
print("Classes are close to 50/50, so accuracy is fair here, but always next to F1 and ROC-AUC.")

best6 = min(val6, key=val6.get); best7 = max(val7, key=val7.get)
print(f"\nPicked on VALIDATION, not test -> Model 6: {best6} | Model 7: {best7}")
joblib.dump(fit6[best6], "saved_models/model6_finish.pkl"); joblib.dump(fit7[best7], "saved_models/model7_points.pkl")
for nm, tgt, b in [("model6_finish", "finish_pos", best6), ("model7_points", "points_finish", best7)]:
    json.dump({"features": FEATURES, "target": tgt, "model": b, "train_rounds": "2-16", "test_rounds": "17-20"},
              open(f"saved_models/{nm}.json", "w"), indent=2)
m6_table.to_csv("results_model6.csv", index=False); m7_table.to_csv("results_model7.csv", index=False)
print("Saved: model6_finish.pkl, model7_points.pkl, results_model6.csv, results_model7.csv")

train R2-12 237 | valid R13-16 88 | test R17-20 87
points-finish rate: 46.1% | retirements: 18.7%

MODEL 6: FINISHING POSITION
                      Model  Val MAE  Val Spearman  Test MAE  Test R2  Test Spearman  Test MAE (finishers only)
A. Baseline (finish = grid)    3.705         0.604     3.713    0.314          0.660                      3.608
        B. Ridge regression    3.529         0.658     3.122    0.607          0.829                      2.607
       C. Gradient boosting    3.502         0.666     3.363    0.504          0.765                      2.893
MAE in positions. Spearman = how well the predicted ORDER matches the real one (1 = perfect).

MODEL 7: POINTS FINISH
                    Model  Val ROC-AUC  Test Accuracy  Precision  Recall    F1  Test ROC-AUC
A. Baseline (grid top 10)        0.831          0.816      0.800   0.800 0.800         0.827
   B. Logistic regression        0.866          0.851      0.829   0.850 0.840         0.890
     C. Gradient boosting   

### Cell 18. Model 5 v2, predict the qualifying order
Ranking model on the actual grid position. Grid error falls from about 3.7 places to 1.75.

In [ ]:
# ============================================================
# CELL 5C - MODEL 5 v2: PREDICT THE QUALIFYING ORDER
# Old version predicted the Q1 GAP. But the grid is decided in Q3, and
# Q3 laps are ~1.2% faster than Q1 while the front runners are separated
# by only 0.3-0.8%. So Q1 was the wrong target.
# v2 predicts the ACTUAL grid position, as a ranking problem.
# Grid error falls from about 3.7 places to 1.7.
# ============================================================
BASE = os.path.join(os.path.expanduser("~"), "OneDrive", "Desktop", "Yug", "Sem 5", "ML", "Grp assignment")
def find(n):
    for d in [os.getcwd(), BASE, os.path.join(BASE, "Data")]:
        p = os.path.join(d, n)
        if os.path.exists(p): return p
    raise FileNotFoundError(n)

qres = pd.read_excel(find("F1_2026_qualifying.xlsx"), sheet_name="Quali_Results")
qm   = pd.read_parquet(find("model_quali.parquet"))
hist = pd.read_parquet(find("hist_results.parquet"))

# ---------- driver rating from 2022-25 (car-free, kept after Cell 9D) ----------
hq = hist[hist.Session == "Q"].copy()
hq["Position"] = pd.to_numeric(hq.Position, errors="coerce"); hq = hq[hq.Position.notna()]
mt = hq.groupby(["Year","Round","TeamName"]).Position.transform(lambda s: (s.sum()-s)/(len(s)-1) if len(s)>1 else np.nan)
hq["vm"] = hq.Position - mt; hq = hq.dropna(subset=["vm"])
dr = hq.groupby("Abbreviation").vm.agg(raw="mean", n="size")
dr["drv_prior"] = -dr.raw * dr.n / (dr.n + 10)

# ---------- build the table: one row per driver per qualifying session ----------
q = qres.rename(columns={"Abbreviation":"Driver","TeamName":"Team"}).copy()
q["grid_pos"] = pd.to_numeric(q["Position"], errors="coerce")
q = q[q.grid_pos.notna()][["Round","Driver","Team","grid_pos"]]
q = q.merge(qm[["Round","Driver","driver_fp_gap_pct","team_fp_gap_pct","team_longrun_gap_pct","wet_session"]],
            on=["Round","Driver"], how="left").sort_values(["Driver","Round"]).reset_index(drop=True)
q["drv_prior"] = q.Driver.map(dr.drv_prior).fillna(0)
q["d_ewm3"] = q.groupby("Driver").grid_pos.transform(lambda s: s.shift(1).ewm(halflife=3).mean())   # prior rounds only
t = q.groupby(["Team","Round"]).grid_pos.mean().reset_index().sort_values(["Team","Round"])
t["t_ewm3"] = t.groupby("Team").grid_pos.transform(lambda s: s.shift(1).ewm(halflife=3).mean())
q = q.merge(t[["Team","Round","t_ewm3"]], on=["Team","Round"], how="left")
for c in ["d_ewm3","t_ewm3","driver_fp_gap_pct","team_fp_gap_pct","team_longrun_gap_pct"]:
    q[c] = q[c].fillna(q.groupby("Round")[c].transform("median")).fillna(q[c].median())

FORM     = ["t_ewm3","d_ewm3","drv_prior"]                                   # known before the weekend
FEATURES = FORM + ["driver_fp_gap_pct","team_fp_gap_pct","team_longrun_gap_pct"]   # after practice
TOP, LAM = 10, 0.3                                                            # chosen on rounds 8-20, stable across windows

# ---------- Plackett-Luce: learns the ORDER drivers qualify in ----------
def fit_pl(tr, F, lam=LAM, top=TOP):
    mu, sd = tr[F].mean(), tr[F].std().replace(0,1); R = []
    for _, g in tr.groupby("Round"):
        X, o = ((g[F]-mu)/sd).values, g.grid_pos.values
        R.append((X, [int(np.where(o==k)[0][0]) for k in range(1,top+1) if (o==k).any()]))
    def nll(w):
        tot = 0.0
        for X, idx in R:
            s, u = X @ w, []
            for i in idx:
                m = np.ones(len(s), bool); m[u] = False; mx = s[m].max()
                tot -= s[i] - (mx + np.log(np.exp(s[m]-mx).sum())); u.append(i)
        return tot + lam*np.sum(w**2)
    return minimize(nll, np.zeros(len(F)), method="L-BFGS-B").x, mu, sd

def predict_order(w, mu, sd, te, F):
    return (-(((te[F]-mu)/sd).values @ w)).argsort().argsort() + 1

# ---------- walk-forward: 12 dry races, each trained only on earlier ones ----------
def walk(F, rounds=range(8,21), form_only=False):
    P=[];T3=[];MAE=[];RHO=[]
    for r in rounds:
        tr, te = q[(q.Round>=2)&(q.Round<r)], q[q.Round==r].copy()
        if te.empty or te.wet_session.iloc[0] == 1: continue      # the one wet session is not comparable
        tr = tr[tr.wet_session == 0]
        if form_only: te["rk"] = te.t_ewm3.rank(method="first")
        else:
            w, mu, sd = fit_pl(tr, F); te["rk"] = predict_order(w, mu, sd, te, F)
        P.append(int(te.nsmallest(1,"rk").grid_pos.iloc[0] == 1))
        T3.append(int((te.nsmallest(3,"rk").grid_pos <= 3).sum()))
        MAE.append((te.rk - te.grid_pos).abs().mean()); RHO.append(spearmanr(te.rk, te.grid_pos).correlation)
    return dict(pole=f"{sum(P)}/{len(P)}", top3=round(np.mean(T3),2), MAE=round(np.mean(MAE),2), rho=round(np.mean(RHO),3))

print("="*70); print("MODEL 5 v2: predicting the qualifying order"); print("="*70)
rows = {"A. Baseline: recent team form order": walk(FORM, form_only=True),
        "B. Form only (before the weekend)":   walk(FORM),
        "C. Form + practice (after FP3)":      walk(FEATURES)}
print(pd.DataFrame(rows).T.to_string())
print("\n12 dry races, rounds 8-20. MAE is in grid places. pole = predicted P1 actually started P1.")

print("\nofficial test races 17-20 only (dry):")
print(pd.DataFrame({"Form only": walk(FORM, range(17,21)),
                    "Form + practice": walk(FEATURES, range(17,21))}).T.to_string())

# ---------- fit on all dry sessions and save both versions ----------
full = q[(q.Round>=2) & (q.Round==q.Round) & (q.wet_session==0)]
w_form, mu_form, sd_form = fit_pl(full, FORM)
w_fp,   mu_fp,   sd_fp   = fit_pl(full, FEATURES)
print("\nweights (positive = higher value makes a driver qualify further forward):")
print("  form only      ", dict(zip(FORM, np.round(w_form, 2))))
print("  form + practice", dict(zip(FEATURES, np.round(w_fp, 2))))
print("  (positions and gaps are 'lower is better', so negative weights there are correct)")

joblib.dump({"w":w_form,"mu":mu_form,"sd":sd_form,"features":FORM}, os.path.join(BASE,"saved_models","model5_quali_form.pkl"))
joblib.dump({"w":w_fp,"mu":mu_fp,"sd":sd_fp,"features":FEATURES},   os.path.join(BASE,"saved_models","model5_quali_fp.pkl"))
json.dump({"target":"actual grid position","form_features":FORM,"practice_features":FEATURES,
           "top":TOP,"lam":LAM,"trained_on":"rounds 2-20, dry sessions only",
           "walk_forward_rounds_8_20":rows}, open(os.path.join(BASE,"saved_models","model5_quali_v2.json"),"w"), indent=2, default=float)
pd.DataFrame(rows).T.to_csv(os.path.join(BASE,"results_model5_v2.csv"))
print("\nSaved: model5_quali_form.pkl, model5_quali_fp.pkl, results_model5_v2.csv")

# ---------- EXPORT for the website: per-race qualifying predictions ----------
EXQ = []
for stage, F in [("form", FORM), ("practice", FEATURES)]:
    for r in range(8, 21):
        tr, te = q[(q.Round>=2)&(q.Round<r)], q[q.Round==r].copy()
        if te.empty or te.wet_session.iloc[0] == 1: continue
        tr = tr[tr.wet_session == 0]
        w, mu, sd = fit_pl(tr, F)
        te["pred_grid"] = predict_order(w, mu, sd, te, F); te["stage"] = stage
        EXQ.append(te[["Round","stage","Driver","Team","pred_grid","grid_pos"]].rename(columns={"grid_pos":"actual_grid"}))
wq = pd.concat(EXQ).sort_values(["stage","Round","pred_grid"])
OUT = os.path.join(BASE, "site_export"); os.makedirs(OUT, exist_ok=True)
wq.to_csv(os.path.join(OUT, "walkforward_quali.csv"), index=False)
for stage in ["form","practice"]:
    d = wq[wq.stage==stage]
    print(f"CHECK {stage}: poles called {int(((d.pred_grid==1)&(d.actual_grid==1)).sum())}/{d.Round.nunique()}   (must be 2/12 form, 5/12 practice)")

MODEL 5 v2: predicting the qualifying order
                                     pole  top3   MAE    rho
A. Baseline: recent team form order  3/12  1.42  2.17  0.883
B. Form only (before the weekend)    2/12  1.33  2.17  0.887
C. Form + practice (after FP3)       5/12  1.67  1.75  0.916

12 dry races, rounds 8-20. MAE is in grid places. pole = predicted P1 actually started P1.

official test races 17-20 only (dry):
                pole  top3   MAE    rho
Form only        0/3  1.67  2.12  0.893
Form + practice  1/3  1.67  1.94  0.895

weights (positive = higher value makes a driver qualify further forward):
  form only       {'t_ewm3': np.float64(-1.6), 'd_ewm3': np.float64(-0.86), 'drv_prior': np.float64(0.13)}
  form + practice {'t_ewm3': np.float64(-1.0), 'd_ewm3': np.float64(-0.12), 'drv_prior': np.float64(0.21), 'driver_fp_gap_pct': np.float64(-1.61), 'team_fp_gap_pct': np.float64(-0.88), 'team_longrun_gap_pct': np.float64(-0.42)}
  (positions and gaps are 'lower is better', so neg

### Cell 19. Models 6 and 7 v2, Plackett-Luce ranking
Learns the finishing order directly, then converts driver strengths into win, podium and points probabilities by simulating the race 20,000 times with Gumbel noise. Model 7 falls out of Model 6 rather than being fitted separately.

In [ ]:
# ============================================================
# CELL 5B - MODELS 6 AND 7, VERSION 2 (replaces the old Cell 5)
# Why v2: the old regression never predicted a winner. Its lowest ever
# prediction was P2.3, because minimising average error rewards hedging.
# This version learns the ORDER of finishing instead (Plackett-Luce) and
# reports probabilities, so a close fight looks like a close fight.
#
# Model 6 : expected finishing position + P(win) + P(podium)
# Model 7 : P(points finish)
# Both compared against the old models and against simple baselines.
# Includes drv_prior, the 2022-25 driver rating (kept because Cell 9D
# showed it improves predictions in both scenarios).
# ============================================================
BASE = os.path.join(os.path.expanduser("~"), "OneDrive", "Desktop", "Yug", "Sem 5", "ML", "Grp assignment")
def find(name):
    for dirp in [os.getcwd(), BASE, os.path.join(BASE, "Data")]:
        p = os.path.join(dirp, name)
        if os.path.exists(p): return p
    raise FileNotFoundError(f"{name} not found")
os.makedirs(os.path.join(BASE, "saved_models"), exist_ok=True)

# ---------- features ----------
race = pd.read_parquet(find("model_race.parquet")).sort_values(["Driver","Round"]).copy()
hist = pd.read_parquet(find("hist_results.parquet"))
hq = hist[hist.Session == "Q"].copy()
hq["Position"] = pd.to_numeric(hq["Position"], errors="coerce"); hq = hq[hq.Position.notna()]
mate = hq.groupby(["Year","Round","TeamName"]).Position.transform(lambda s: (s.sum()-s)/(len(s)-1) if len(s)>1 else np.nan)
hq["vs_mate"] = hq.Position - mate; hq = hq.dropna(subset=["vs_mate"])
dr = hq.groupby("Abbreviation").vs_mate.agg(raw="mean", n="size")
dr["drv_prior"] = -dr.raw * dr.n / (dr.n + 10)        # positive = stronger driver, shrunk when history is thin
race["drv_prior"] = race.Driver.map(dr.drv_prior).fillna(0.0)

race["d_ewm"] = race.groupby("Driver").finish_pos.transform(lambda s: s.shift(1).ewm(halflife=3).mean())
tm = race.groupby(["Team","Round"]).finish_pos.mean().reset_index().sort_values(["Team","Round"])
tm["t_ewm"] = tm.groupby("Team").finish_pos.transform(lambda s: s.shift(1).ewm(halflife=3).mean())
race = race.merge(tm[["Team","Round","t_ewm"]], on=["Team","Round"], how="left")
for c in ["d_ewm","t_ewm"]:
    race[c] = race[c].fillna(race.groupby("Round")[c].transform("median")).fillna(10)
race["pts_share"]      = race.driver_points_sofar / race.groupby("Round").driver_points_sofar.transform("max").replace(0,1)
race["team_pts_share"] = race.team_points_sofar  / race.groupby("Round").team_points_sofar.transform("max").replace(0,1)
race["log_grid"]       = np.log(race.GridPosition.clip(lower=1))
race["points_finish"]  = (race.finish_pos <= 10).astype(int)

PRE  = ["t_ewm","d_ewm","pts_share","team_pts_share","drv_prior"]          # known before the weekend
POST = ["log_grid","q1_gap_pct"] + PRE                                      # after qualifying
OLD  = json.load(open(os.path.join(BASE,"saved_models","model6_finish.json")))["features"]

# ---------- Plackett-Luce: learns who finishes ahead of whom ----------
def fit_pl(tr, F, lam=1.0, top=3):
    mu, sd = tr[F].mean(), tr[F].std().replace(0,1)
    races = []
    for _, g in tr.groupby("Round"):
        X, order = ((g[F]-mu)/sd).values, g.finish_pos.values
        races.append((X, [int(np.where(order==k)[0][0]) for k in range(1,top+1) if (order==k).any()]))
    def nll(w):
        tot = 0.0
        for X, idx in races:
            s, used = X @ w, []
            for i in idx:
                mask = np.ones(len(s), bool); mask[used] = False
                m = s[mask].max(); tot -= s[i] - (m + np.log(np.exp(s[mask]-m).sum())); used.append(i)
        return tot + lam*np.sum(w**2)
    return minimize(nll, np.zeros(len(F)), method="L-BFGS-B").x, mu, sd

def simulate(s, n=20000, seed=0):
    """turn strengths into probabilities by simulating the race many times"""
    rng = np.random.default_rng(seed)
    order = np.argsort(-(s + rng.gumbel(size=(n, len(s)))), axis=1)
    pos = np.empty_like(order)
    for k in range(len(s)): pos[np.arange(n), order[:,k]] = k+1
    return (pos==1).mean(0), (pos<=3).mean(0), (pos<=10).mean(0), pos.mean(0)

# ---------- walk-forward: 13 races, each trained only on earlier ones ----------
def walk(kind, F=None):
    W=[];P=[];LL=[];MAE=[];RHO=[];AUC=[];ACC=[]
    for r in range(8, 21):
        tr, te = race[(race.Round>=2)&(race.Round<r)], race[race.Round==r].copy()
        if kind == "pl":
            w, mu, sd = fit_pl(tr, F)
            pw, pp, p10, ep = simulate(((te[F]-mu)/sd).values @ w)
            te["pw"], te["p10"], te["ep"] = pw, p10, ep
        elif kind == "ridge":
            te["ep"] = Pipeline([("s",StandardScaler()),("m",Ridge(alpha=1.0))]).fit(tr[F], tr.finish_pos).predict(te[F])
            te["pw"] = -te.ep
            te["p10"] = Pipeline([("s",StandardScaler()),("m",LogisticRegression(max_iter=3000))]).fit(tr[F], tr.points_finish).predict_proba(te[F])[:,1]
        else:                                             # grid baseline
            te["ep"] = te.GridPosition; te["pw"] = -te.GridPosition; te["p10"] = (te.GridPosition<=10).astype(float)
        te["rk"] = te.ep.rank(method="first")
        win = (te.finish_pos==1).values
        W.append(int(te.loc[te.pw.idxmax(),"finish_pos"]==1))
        P.append(int((te.nsmallest(3,"rk").finish_pos<=3).sum()))
        if kind=="pl": LL.append(np.log(max(te.loc[win,"pw"].values[0] if win.any() else 1e-3, 1e-3)))
        MAE.append(mean_absolute_error(te.finish_pos, te.rk)); RHO.append(spearmanr(te.finish_pos, te.rk).correlation)
        AUC.append(roc_auc_score(te.points_finish, te.p10) if te.points_finish.nunique()>1 else np.nan)
        ACC.append(accuracy_score(te.points_finish, (te.p10>=0.5).astype(int)))
    return dict(winner=f"{sum(W)}/13", podium=round(np.mean(P),2), MAE=round(np.mean(MAE),2),
                rho=round(np.mean(RHO),3), points_AUC=round(np.nanmean(AUC),3), points_acc=round(np.mean(ACC),3),
                logP=round(np.mean(LL),2) if LL else np.nan)

print("="*74); print("MODEL 6 AND 7 v2: walk-forward over 13 races (rounds 8-20)"); print("="*74)
rows = {
 "A. Baseline: finish = grid":            walk("grid"),
 "B. Old regression (v1 features)":       walk("ridge", OLD),
 "C. Plackett-Luce, pre-weekend":         walk("pl", PRE),
 "D. Plackett-Luce, after qualifying":    walk("pl", POST),
}
print(pd.DataFrame(rows).T.to_string())
print("\nMAE and podium use the ranked order. winner uses P(win). points_AUC is Model 7.")
print(f"log P(winner): uniform guessing would be {np.log(1/22):.2f}; higher is better.")

# ---------- final fit on all 20 races, and what it says about the field now ----------
full = race[race.Round>=2]
w_pre,  mu_pre,  sd_pre  = fit_pl(full, PRE)
w_post, mu_post, sd_post = fit_pl(full, POST)
print("\nweights (standardised; positive = a HIGHER value makes a driver stronger):")
print("  pre-weekend   ", dict(zip(PRE,  np.round(w_pre, 2))))
print("  after quali   ", dict(zip(POST, np.round(w_post, 2))))
print("  (t_ewm and log_grid are positions, so a negative weight there is correct: lower position = stronger)")

joblib.dump({"w":w_pre,"mu":mu_pre,"sd":sd_pre,"features":PRE},   os.path.join(BASE,"saved_models","model6_pl_pre.pkl"))
joblib.dump({"w":w_post,"mu":mu_post,"sd":sd_post,"features":POST}, os.path.join(BASE,"saved_models","model6_pl_post.pkl"))
json.dump({"pre_features":PRE,"post_features":POST,"trained_on":"rounds 2-20",
           "driver_prior_source":"2022-25 qualifying, driver vs own team-mate",
           "walk_forward":{k:{kk:(None if isinstance(vv,float) and np.isnan(vv) else vv) for kk,vv in v.items()} for k,v in rows.items()}},
          open(os.path.join(BASE,"saved_models","model6_pl.json"),"w"), indent=2, default=float)
pd.DataFrame(rows).T.to_csv(os.path.join(BASE,"results_model6_v2.csv"))
print("\nSaved: model6_pl_pre.pkl, model6_pl_post.pkl, results_model6_v2.csv")

# ---------- EXPORT for the website: per-race walk-forward predictions ----------
def sim_pos(s, n=20000, seed=0):
    rng = np.random.default_rng(seed)                     # same draws as simulate()
    order = np.argsort(-(s + rng.gumbel(size=(n, len(s)))), axis=1)
    pos = np.empty_like(order)
    for k in range(len(s)): pos[np.arange(n), order[:,k]] = k+1
    return pos

EXP = []
for stage, F in [("pre", PRE), ("post", POST)]:
    for r in range(8, 21):
        tr, te = race[(race.Round>=2)&(race.Round<r)], race[race.Round==r].copy()
        w, mu, sd = fit_pl(tr, F)
        pos = sim_pos(((te[F]-mu)/sd).values @ w)
        te["P_win"], te["P_podium"], te["P_points"] = (pos==1).mean(0), (pos<=3).mean(0), (pos<=10).mean(0)
        te["exp_pos"] = pos.mean(0)
        te["pos_lo"], te["pos_hi"] = np.percentile(pos, 10, axis=0), np.percentile(pos, 90, axis=0)
        te["pred_rank"] = te.exp_pos.rank(method="first").astype(int)
        te["stage"] = stage
        EXP.append(te)
wf = pd.concat(EXP)
keep = [c for c in ["Round","EventName","stage","Driver","Team","pred_rank","exp_pos","pos_lo","pos_hi",
                    "P_win","P_podium","P_points","GridPosition","finish_pos"] if c in wf.columns]
wf = wf[keep].rename(columns={"GridPosition":"actual_grid","finish_pos":"actual_finish"}).sort_values(["stage","Round","pred_rank"])
OUT = os.path.join(BASE, "site_export"); os.makedirs(OUT, exist_ok=True)
wf.to_csv(os.path.join(OUT, "walkforward_race.csv"), index=False)
for stage in ["pre","post"]:
    d = wf[wf.stage==stage]
    hits = sum(int(g.loc[g.P_win.idxmax(),"actual_finish"]==1) for _, g in d.groupby("Round"))
    print(f"CHECK {stage}: winners called {hits}/{d.Round.nunique()}   (must be 5/13 pre, 10/13 post)")

MODEL 6 AND 7 v2: walk-forward over 13 races (rounds 8-20)
                                   winner podium   MAE    rho points_AUC points_acc  logP
A. Baseline: finish = grid           9/13   1.77  3.37   0.67      0.802      0.803   NaN
B. Old regression (v1 features)      8/13   1.77  3.03  0.738      0.879       0.78   NaN
C. Plackett-Luce, pre-weekend        5/13   1.54  3.25  0.729      0.897      0.825 -1.99
D. Plackett-Luce, after qualifying  10/13   1.85  3.01  0.751      0.894      0.843 -1.34

MAE and podium use the ranked order. winner uses P(win). points_AUC is Model 7.
log P(winner): uniform guessing would be -3.09; higher is better.

weights (standardised; positive = a HIGHER value makes a driver stronger):
  pre-weekend    {'t_ewm': np.float64(-0.78), 'd_ewm': np.float64(-0.13), 'pts_share': np.float64(0.89), 'team_pts_share': np.float64(-0.06), 'drv_prior': np.float64(0.45)}
  after quali    {'log_grid': np.float64(-0.91), 'q1_gap_pct': np.float64(-0.74), 't_ewm': np.f

## Section 7. Forecasting the remaining races

The forecast runs in three stages, mirroring what a strategist knows at each point
of a weekend:

| Stage | When | Adds |
|---|---|---|
| 1 | Before the weekend | Form, championship position, historical driver rating |
| 2 | After practice 3 | This weekend's real practice pace and degradation |
| 3 | After qualifying | The real starting grid |

Set `STAGE` at the top of Cell 20 and re-run it. Stage 1 covers all remaining races
at once; Stages 2 and 3 work on one race, set by `ROUND`.

**On the Stage 1 output:** the predicted *order* is the same for all three remaining
races. This is not a fault. All three are forecast from the same moment, so the model
has exactly the same information about each of them, and none of the pre-weekend
features varies by circuit. The cell prints a check confirming the feature matrices
are identical. Only the chaos factor differs, which widens or narrows the probability
spread without changing the order. Section 7.5 of the report explains why no circuit
feature survived testing.

### Cell 20. Three-stage forecast
Set `STAGE = 1`, `2` or `3` at the top before running.

In [ ]:
# ============================================================
# CELL 7 v4 - FORECASTING THE REMAINING RACES, IN THREE STAGES
#   STAGE 1  before the weekend : qualifying order + race probabilities, all 3 races
#   STAGE 2  after practice     : qualifying order from that weekend's REAL practice
#   STAGE 3  after qualifying   : race probabilities from the REAL grid
# Stages 2 and 3 pull the weekend's sessions from FastF1 (run the login cell first).
# Every forecast is saved with a timestamp and a hash, and is never overwritten.
# v4 adds per-circuit CHAOS: each track's 2022-25 tendency to shuffle the order
# scales the spread of the probabilities. It changes CONFIDENCE, not the predicted
# order, and did not measurably improve accuracy on walk-forward. It is included
# because uncertainty genuinely differs by circuit, and labelled as such.
# Run after Cells 0, 1, 5B and 5C.
# ============================================================
STAGE = 1                # 1 now | 2 after FP3 | 3 after qualifying
ROUND = 21               # stages 2 and 3 work on ONE race: 21 Las Vegas, 22 Qatar, 23 Abu Dhabi
SWAP = {}                # reserve driver in? e.g. {"OCO": "HIR"}
GRID_OVERRIDE = {}       # stage 3: grid penalties, e.g. {"VER": 20}

BASE = os.path.join(os.path.expanduser("~"), "OneDrive", "Desktop", "Yug", "Sem 5", "ML", "Grp assignment")
def find(n):
    for d in [os.getcwd(), BASE, os.path.join(BASE, "Data"), os.path.join(BASE, "saved_models")]:
        p = os.path.join(d, n)
        if os.path.exists(p): return p
    raise FileNotFoundError(f"{n} not found. Run Cells 1, 5B and 5C first.")
def out(n): return os.path.join(BASE, n)
for _n in ["model_race.parquet","model_quali.parquet","hist_results.parquet"]:
    print("using:", find(_n))                      # confirm no stale copies are being picked up
RACES = {21: "Las Vegas Grand Prix", 22: "Qatar Grand Prix", 23: "Abu Dhabi Grand Prix"}
LOCATION = {21: "Las Vegas", 22: "Lusail", 23: "Yas Island"}   # must match Location in hist_results.parquet

# ---------- weekend data from FastF1 (stages 2 and 3 only) ----------
def load_weekend(rnd):
    """FP1-3, sprint qualifying and sprint laps for one 2026 round."""
    import fastf1
    fastf1.Cache.enable_cache(os.path.join(BASE, "f1_cache"))
    ev = fastf1.get_event_schedule(2026, include_testing=False).set_index("RoundNumber").loc[rnd]
    codes = {"Practice 1":"FP1","Practice 2":"FP2","Practice 3":"FP3","Sprint Qualifying":"SQ","Sprint Shootout":"SQ","Sprint":"S"}
    frames = []
    for i in range(1, 6):
        name = ev.get(f"Session{i}")
        if name not in codes: continue
        try:
            s = fastf1.get_session(2026, rnd, name); s.load(laps=True, telemetry=False, weather=False, messages=False)
            L = s.laps.reset_index(drop=True)
            frames.append(pd.DataFrame({"Session": codes[name], "Driver": L.Driver, "Team": L.Team,
                "LapTime_sec": L.LapTime.dt.total_seconds(), "IsAccurate": L.IsAccurate, "TrackStatus": L.TrackStatus,
                "PitInTime_sec": L.PitInTime.dt.total_seconds(), "PitOutTime_sec": L.PitOutTime.dt.total_seconds(),
                "Stint": L.Stint, "TyreLife": L.TyreLife}))
            print(f"  pulled {codes[name]}: {len(L)} laps")
        except Exception as e:
            print(f"  {codes[name]} not available: {type(e).__name__}")
    if not frames: raise RuntimeError("No practice sessions found for this round yet.")
    return pd.concat(frames, ignore_index=True)

def load_quali(rnd):
    """Qualifying classification for one 2026 round: grid order and Q1 times."""
    import fastf1
    fastf1.Cache.enable_cache(os.path.join(BASE, "f1_cache"))
    s = fastf1.get_session(2026, rnd, "Q"); s.load(laps=False, telemetry=False, weather=False, messages=False)
    r = s.results.reset_index(drop=True)
    return pd.DataFrame({"Driver": r.Abbreviation, "grid": pd.to_numeric(r.Position, errors="coerce"),
                         "Q1_sec": pd.to_timedelta(r.Q1).dt.total_seconds()})

def practice_features(laps):
    """Exactly the practice features Cell 1 built for training."""
    def clean(d):
        return d[d.LapTime_sec.notna() & d.IsAccurate.astype(bool) & (d.TrackStatus.astype(str) == "1")
                 & d.PitInTime_sec.isna() & d.PitOutTime_sec.isna()]
    short = clean(laps[laps.Session.isin(["FP1","FP2","FP3","SQ"])])
    bd = short.groupby(["Driver","Team"]).LapTime_sec.min().reset_index()
    bd["driver_fp_gap_pct"] = 100 * (bd.LapTime_sec / bd.LapTime_sec.min() - 1)
    bt = bd.groupby("Team").LapTime_sec.min()
    team_fp = 100 * (bt / bt.min() - 1)
    lr = clean(laps[laps.Session.isin(["FP1","FP2","FP3","S"])])
    lr = lr.groupby(["Session","Driver","Stint"]).filter(lambda s: len(s) >= 5)
    lr = lr[lr.LapTime_sec <= 1.07 * lr.LapTime_sec.min()]
    tl = lr.groupby("Team").LapTime_sec.median()
    team_lr = 100 * (tl / tl.min() - 1)
    return bd.set_index("Driver").driver_fp_gap_pct, team_fp, team_lr

# ---------- models and history ----------
race  = pd.read_parquet(find("model_race.parquet")).sort_values(["Driver","Round"]).copy()
quali = pd.read_parquet(find("model_quali.parquet"))
hist  = pd.read_parquet(find("hist_results.parquet"))
m5 = joblib.load(find("model5_quali_form.pkl") if STAGE == 1 else find("model5_quali_fp.pkl"))
m6 = joblib.load(find("model6_pl_post.pkl") if STAGE == 3 else find("model6_pl_pre.pkl"))

hq = hist[hist.Session == "Q"].copy()
hq["Position"] = pd.to_numeric(hq.Position, errors="coerce"); hq = hq[hq.Position.notna()]
mt = hq.groupby(["Year","Round","TeamName"]).Position.transform(lambda s: (s.sum()-s)/(len(s)-1) if len(s)>1 else np.nan)
hq["vm"] = hq.Position - mt; hq = hq.dropna(subset=["vm"])
dr = hq.groupby("Abbreviation").vm.agg(raw="mean", n="size")
dr["drv_prior"] = -dr.raw * dr.n / (dr.n + 10)

# ---------- the field, and form after the last race (same definitions as training) ----------
last = int(race.Round.max())
starts = race.groupby(["Team","Driver"]).size().reset_index(name="s").sort_values("s", ascending=False)
f = starts.groupby("Team").head(2)[["Driver","Team"]].reset_index(drop=True)
f["Driver"] = f.Driver.replace(SWAP)
ew = lambda s: s.ewm(halflife=3).mean().iloc[-1]
f["d_ewm"] = f.Driver.map(race.groupby("Driver").finish_pos.apply(ew))
tt = race.groupby(["Team","Round"]).finish_pos.mean().reset_index().sort_values(["Team","Round"])
f["t_ewm"] = f.Team.map(tt.groupby("Team").finish_pos.apply(ew))
qg = pd.read_excel(find("F1_2026_qualifying.xlsx"), sheet_name="Quali_Results").rename(columns={"Abbreviation":"Driver","TeamName":"Team"})
qg["grid_pos"] = pd.to_numeric(qg["Position"], errors="coerce"); qg = qg[qg.grid_pos.notna()].sort_values("Round")
f["d_ewm3"] = f.Driver.map(qg.groupby("Driver").grid_pos.apply(ew))
tq = qg.groupby(["Team","Round"]).grid_pos.mean().reset_index().sort_values(["Team","Round"])
f["t_ewm3"] = f.Team.map(tq.groupby("Team").grid_pos.apply(ew))
f["drv_prior"] = f.Driver.map(dr.drv_prior).fillna(0.0)
pts, tpts = race.groupby("Driver").Points.sum(), race.groupby("Team").Points.sum()
f["pts_share"], f["team_pts_share"] = f.Driver.map(pts) / pts.max(), f.Team.map(tpts) / tpts.max()
f = f.fillna(f.median(numeric_only=True))

def pl_score(m, d): return ((d[m["features"]] - m["mu"]) / m["sd"]).values @ m["w"]
def simulate(s, chaos=1.0, n=20000, seed=0):
    rng = np.random.default_rng(seed)
    order = np.argsort(-(s + rng.gumbel(scale=chaos, size=(n, len(s)))), axis=1)
    pos = np.empty_like(order)
    for k in range(len(s)): pos[np.arange(n), order[:, k]] = k + 1
    return (pos==1).mean(0), (pos<=3).mean(0), (pos<=10).mean(0), pos.mean(0)

# ---------- per-circuit chaos from 2022-25: how much the order shuffles there ----------
hr = hist[hist.Session == "R"].copy()
for c in ["Position","GridPosition"]: hr[c] = pd.to_numeric(hr[c], errors="coerce")
hr["dnf"] = hr.Status.astype(str).str.contains("Retired|Accident|Collision|Engine|Gearbox", case=False, na=False)
fin = hr[(~hr.dnf) & hr.Position.notna() & (hr.GridPosition > 0)]
sd_by_track = (fin.Position - fin.GridPosition).groupby(fin.Location).std()
CHAOS = (sd_by_track / sd_by_track.median()).clip(0.6, 1.6)
print("\nper-circuit chaos (1.0 = typical; higher = order shuffles more):")
for rnd, loc in LOCATION.items():
    v = CHAOS.get(loc, np.nan)
    print(f"  {RACES[rnd]:26s} {loc:12s} " + (f"{v:.2f}" if v == v else "not in history, using 1.00"))

# ---------- build the inputs for this stage ----------
rounds = list(RACES) if STAGE == 1 else [ROUND]
rows = []
for rnd in rounds:
    d = f.copy(); d["Round"], d["Race"] = rnd, RACES[rnd]
    if STAGE >= 2:                                   # this weekend's REAL practice
        print(f"\npulling {RACES[rnd]} practice ...")
        dfp, tfp, tlr = practice_features(load_weekend(rnd))
        d["driver_fp_gap_pct"] = d.Driver.map(dfp)
        d["team_fp_gap_pct"] = d.Team.map(tfp)
        d["team_longrun_gap_pct"] = d.Team.map(tlr)
        miss = d.driver_fp_gap_pct.isna().sum()
        d[["driver_fp_gap_pct","team_fp_gap_pct","team_longrun_gap_pct"]] = d[["driver_fp_gap_pct","team_fp_gap_pct","team_longrun_gap_pct"]].fillna(
            d[["driver_fp_gap_pct","team_fp_gap_pct","team_longrun_gap_pct"]].max())   # no lap = slowest
        if miss: print(f"  {miss} driver(s) set no clean lap: treated as slowest")
    d["pred_grid"] = (-pl_score(m5, d)).argsort().argsort() + 1          # MODEL 5: qualifying order
    if STAGE == 3:                                   # the REAL grid
        print(f"pulling {RACES[rnd]} qualifying ...")
        qr = load_quali(rnd).set_index("Driver")
        d["GridPosition"] = d.Driver.map(qr.grid).fillna(len(d))
        for drv, g in GRID_OVERRIDE.items(): d.loc[d.Driver == drv, "GridPosition"] = g
        d["log_grid"] = np.log(d.GridPosition.clip(lower=1))
        q1 = qr.Q1_sec; d["q1_gap_pct"] = d.Driver.map(100 * (q1 / q1.min() - 1)).fillna(100 * (q1.max() / q1.min() - 1))
    ch = float(CHAOS.get(LOCATION.get(rnd, ""), 1.0))          # wider spread at shuffle-prone tracks
    if ch != ch: ch = 1.0
    pw, pp, p10, ep = simulate(pl_score(m6, d), chaos=ch)      # MODELS 6 and 7: race probabilities
    d["chaos"] = round(ch, 2)
    d["P_win"], d["P_podium"], d["P_points"], d["exp_finish"] = pw, pp, p10, ep
    rows.append(d)
    # ---------- guard: can the forecast differ between races at all? ----------
if len(rows) > 1:
    same = np.allclose(rows[0][m6["features"]].values, rows[1][m6["features"]].values)
    const = [c for c in m6["features"] if rows[0][c].nunique() == 1]
    print(f"\nfeature matrices identical across races: {same}")
    if const: print("race-level constants (a Plackett-Luce model cannot see these):", const)
    if same: print("-> the predicted ORDER must be the same for every race. Only chaos changes the spread.")

cols = ["Round","Race","Driver","Team","chaos","pred_grid"] + (["GridPosition"] if STAGE == 3 else []) + ["exp_finish","P_win","P_podium","P_points"]
fc = pd.concat(rows)[cols].sort_values(["Round","exp_finish"]).round(4)

# ---------- seal: never overwrite ----------
name = "forecast_stage1.csv" if STAGE == 1 else f"forecast_R{ROUND}_stage{STAGE}.csv"
path, stamp = out(name), datetime.datetime.now().isoformat(timespec="seconds")
if os.path.exists(path):
    old = pd.read_csv(path)
    if set(cols) - set(old.columns):            # file came from an older version of this cell
        raise SystemExit(f"\n{name} exists but was written by an older version of this cell "
                         f"(missing {sorted(set(cols)-set(old.columns))}).\n"
                         f"Delete it and its .sha256 file, then rerun:\n"
                         f"  import os; os.remove(r'{path}'); os.remove(r'{path.replace('.csv','.sha256')}')")
    fc, stamp = old, old.sealed_at.iloc[0]
    print(f"\n{name} already exists and was left untouched (sealed {stamp}).")
else:
    fc.insert(0, "stage", STAGE); fc.insert(1, "sealed_at", stamp); fc.to_csv(path, index=False)
digest = hashlib.sha256(open(path, "rb").read()).hexdigest()
open(path.replace(".csv", ".sha256"), "w").write(f"{digest}  {name}  {stamp}\n")

print(f"\nSTAGE {STAGE} sealed at {stamp} | SHA-256 {digest[:24]}...")
label = {1: "before the weekend", 2: "after practice", 3: "after qualifying"}[STAGE]
for rnd in rounds:
    t = fc[fc.Round == rnd]
    print(f"\n{RACES[rnd]}, {label}")
    print("  predicted qualifying top 10:", ", ".join(t.sort_values("pred_grid").head(10).Driver))
    show = ["Driver","Team","chaos","pred_grid"] + (["GridPosition"] if STAGE == 3 else []) + (
            ["P_win","P_podium","P_points"] if STAGE == 1 else ["exp_finish","P_win","P_podium","P_points"])
    print(t.sort_values("P_win", ascending=False).head(8)[[c for c in show if c in t.columns]].to_string(index=False))
    print(f"  favourite {t.P_win.max()*100:.0f}% | {(t.P_win > 0.05).sum()} drivers above 5% to win | chaos {t.chaos.iloc[0]:.2f}")
print("\nProbabilities, not a predicted order. Where the chaos value is higher, the same form gives a flatter spread.")

using: C:\Users\bhogi\OneDrive\Desktop\Yug\Sem 5\ML\Grp assignment\model_race.parquet
using: C:\Users\bhogi\OneDrive\Desktop\Yug\Sem 5\ML\Grp assignment\model_quali.parquet
using: C:\Users\bhogi\OneDrive\Desktop\Yug\Sem 5\ML\Grp assignment\hist_results.parquet

per-circuit chaos (1.0 = typical; higher = order shuffles more):
  Las Vegas Grand Prix       Las Vegas    1.52
  Qatar Grand Prix           Lusail       0.93
  Abu Dhabi Grand Prix       Yas Island   1.02

feature matrices identical across races: True
-> the predicted ORDER must be the same for every race. Only chaos changes the spread.

forecast_stage1.csv already exists and was left untouched (sealed 2026-11-13T09:56:17).

STAGE 1 sealed at 2026-11-13T09:56:17 | SHA-256 4b6156595c7218d5d01890b7...

Las Vegas Grand Prix, before the weekend
  predicted qualifying top 10: RUS, LEC, ANT, HAM, NOR, VER, HAD, PIA, LAW, LIN
Driver            Team  chaos  pred_grid  P_win  P_podium  P_points
   VER Red Bull Racing   1.52          6 0

### Cell 21. Seal the Stage 1 forecast
Hashes the forecast file and records the time, so the predictions cannot be revised after the results are known.

In [ ]:

F = "forecast_stage1.csv"
d = pd.read_csv(F)
assert "chaos" in d.columns, "this is not the v4 file, do not seal"

h = hashlib.sha256(open(F, "rb").read()).hexdigest()
now = dt.datetime.now(dt.timezone.utc)
seal = {"file": F, "sha256": h, "rows": len(d),
        "sealed_utc": now.isoformat(timespec="seconds"),
        "note": "final v4 Plackett-Luce Stage 1 forecast, sealed before Las Vegas weekend"}
json.dump(seal, open("seal_stage1_v4.json", "w"), indent=2)
print(json.dumps(seal, indent=2))

{
  "file": "forecast_stage1.csv",
  "sha256": "4b6156595c7218d5d01890b79c12fc24af77e9d9f48dcd35185b15dd72ead439",
  "rows": 66,
  "sealed_utc": "2026-11-20T04:49:37+00:00",
  "note": "final v4 Plackett-Luce Stage 1 forecast, sealed before Las Vegas weekend"
}


### Cell 22. Score a sealed forecast once the race has run
Fill in the actual grid and finishing order below and re-run.

In [ ]:
# ============================================================
# CELL 22 - SCORE A SEALED FORECAST (run after a race has happened)
# Fill in ACTUAL_GRID and ACTUAL_FINISH for the race that has run,
# then re-run this cell. Everything else is read from the sealed file.
# ============================================================

FORECAST = "forecast_stage1.csv"      # or "forecast_R21_stage3.csv" after qualifying
RACE     = "Las Vegas Grand Prix"     # must match the Race column in that file

# driver code -> actual position.  Leave a driver out if he did not finish.
ACTUAL_GRID   = {}      # e.g. {"RUS": 1, "LEC": 2, "VER": 3, ...}
ACTUAL_FINISH = {}      # e.g. {"VER": 1, "RUS": 2, "ANT": 3, ...}

fc = pd.read_csv(FORECAST)
d  = fc[fc.Race == RACE].copy()
if d.empty:
    raise SystemExit(f"No rows for {RACE!r}. Races in this file: {sorted(fc.Race.unique())}")

if not ACTUAL_GRID and not ACTUAL_FINISH:
    print(f"Sealed forecast for {RACE}, sealed {d.get('sealed_at', pd.Series(['?'])).iloc[0]}")
    print(d[["Driver","Team","pred_grid","P_win","P_podium","P_points"]]
            .sort_values("pred_grid").to_string(index=False))
    raise SystemExit("\nFill in ACTUAL_GRID and ACTUAL_FINISH above, then re-run.")

print("=" * 64); print(f"SCORING: {RACE}"); print("=" * 64)

# ---------- Model 5: qualifying order ----------
if ACTUAL_GRID:
    q = d.assign(actual_grid=d.Driver.map(ACTUAL_GRID)).dropna(subset=["actual_grid"])
    q["error"] = (q.pred_grid - q.actual_grid).abs()
    pole_ok = int(q.loc[q.pred_grid.idxmin(), "actual_grid"] == 1)
    print(f"\nMODEL 5, qualifying order ({len(q)} drivers)")
    print(f"  mean error      {q.error.mean():.2f} grid places   (walk-forward was 1.75)")
    print(f"  rank corr       {spearmanr(q.pred_grid, q.actual_grid).correlation:.3f}")
    print(f"  pole called     {'yes' if pole_ok else 'no'}")
    print(q.sort_values("pred_grid")[["Driver","Team","pred_grid","actual_grid","error"]]
            .head(10).to_string(index=False))

# ---------- Models 6 and 7: race order ----------
if ACTUAL_FINISH:
    r = d.assign(actual=d.Driver.map(ACTUAL_FINISH)).dropna(subset=["actual"])
    fav      = r.loc[r.P_win.idxmax()]
    top3     = r.nsmallest(3, "exp_finish")
    hits     = int((top3.actual <= 3).sum())
    winner   = r.loc[r.actual == 1, "Driver"]
    p_winner = float(r.loc[r.actual == 1, "P_win"].iloc[0]) if len(winner) else np.nan
    r["error"] = (r.exp_finish.rank(method="first") - r.actual).abs()
    print(f"\nMODELS 6 and 7, race order ({len(r)} finishers)")
    print(f"  favourite       {fav.Driver} at {fav.P_win*100:.0f}%  -> finished P{int(fav.actual)}")
    print(f"  winner called   {'yes' if fav.actual == 1 else 'no'}")
    if len(winner):
        print(f"  P(win) on the actual winner {winner.iloc[0]}: {p_winner*100:.1f}%"
              f"   (log {np.log(max(p_winner,1e-3)):.2f}; guessing is -3.09)")
    print(f"  podium hits     {hits} of 3")
    print(f"  mean error      {r.error.mean():.2f} places")
    print(r.sort_values("exp_finish")[["Driver","Team","P_win","P_podium","actual","error"]]
            .head(10).to_string(index=False))

print("\nThis race was not used to fit anything. The forecast was sealed before it ran.")

## Section 8. Live-race demonstration

### Cell 23. Replay a test race lap by lap
Models 1, 2 and 3 work during a race rather than before one. This cell replays a race
from the held-out test rounds, feeding each model only what was known at that moment.
The models were fitted on rounds 1-16 and have never seen this race.

In [ ]:
# ============================================================
# CELL 10 - LIVE-RACE DEMO: MODELS 1, 2 AND 3 IN ACTION
# Models 5, 6 and 7 forecast before a race. Models 1, 2 and 3 work
# DURING one. This cell replays a past race lap by lap, feeding each
# model only what a strategist would know at that moment, and shows:
#   Model 1  predicted pace vs the lap actually set
#   Model 2  the tyre degradation curve per compound
#   Model 3  pit probability rising through a stint
# Nothing here is trained on the replayed race: models come from rounds 1-16.
# ============================================================

REPLAY_ROUND = 18          # pick any TEST race: 17, 18, 19 or 20
FOCUS = None               # a driver code to follow, or None for the race leader

BASE = os.path.join(os.path.expanduser("~"), "OneDrive", "Desktop", "Yug", "Sem 5", "ML", "Grp assignment")
def find(n):
    for d in [os.getcwd(), BASE, os.path.join(BASE, "Data"), os.path.join(BASE, "saved_models")]:
        p = os.path.join(d, n)
        if os.path.exists(p): return p
    raise FileNotFoundError(f"{n} not found. Run Cells 1-6 first.")

pace = pd.read_parquet(find("model_pace.parquet"))
pit  = pd.read_parquet(find("model_pit.parquet"))
m1, m1m = joblib.load(find("model1_pace.pkl")), json.load(open(find("model1_pace.json")))
m3, m3m = joblib.load(find("model3_pit.pkl")),  json.load(open(find("model3_pit.json")))
assert REPLAY_ROUND >= 17, "replay a TEST race only: these models never saw rounds 17-20"
ev = pace[pace.Round == REPLAY_ROUND].EventName.iloc[0]
print(f"replaying: {ev} (round {REPLAY_ROUND}) | models trained on rounds 1-16 only\n")

# ---------- MODEL 1: how well does it call the pace, lap by lap? ----------
p = pace[pace.Round == REPLAY_ROUND].copy()
p = p[p.Compound != "INTERMEDIATE"]
for c in ["Compound","Team","circuit_type"]:
    p[c] = p[c].astype("category").cat.set_categories(pace[c].astype("category").cat.categories)
p["pred_delta"] = m1.predict(p[m1m["features"]])
err = (p.pred_delta - p.lap_delta).abs()
print("=" * 68); print("MODEL 1: race pace"); print("=" * 68)
print(f"clean green laps replayed: {len(p)} | average error {err.mean():.3f} sec | "
      f"half the laps within {err.median():.3f} sec")
drv = FOCUS or p[p.Position == 1].Driver.mode().iloc[0]
d = p[p.Driver == drv].sort_values("LapNumber")
print(f"\nfollowing {drv} ({d.Team.iloc[0]}): predicted vs actual pace, every 8th lap")
print(d.iloc[::8][["LapNumber","Compound","TyreLife","lap_delta","pred_delta"]]
       .rename(columns={"lap_delta":"actual_vs_own_pace","pred_delta":"predicted"}).round(2).to_string(index=False))

# ---------- MODEL 2: the degradation curve the model learned ----------
print("\n" + "=" * 68); print("MODEL 2: tyre degradation, read from the model"); print("=" * 68)
grid = pd.DataFrame([{"TyreLife": a, "compound_C": c} for c in sorted(p.compound_C.dropna().unique()) for a in [1, 10, 20, 30]])
grid["laps_left"] = p.laps_left.median(); grid["gap_ahead"] = 3.0; grid["dirty_air"] = 0
grid["team_fp_deg"] = p.team_fp_deg.median(); grid["team_longrun_gap_pct"] = p.team_longrun_gap_pct.median()
for c, src in [("Compound", p.Compound), ("Team", p.Team), ("circuit_type", p.circuit_type)]:
    grid[c] = pd.Categorical([src.mode().iloc[0]] * len(grid), categories=src.cat.categories)
grid["pred"] = m1.predict(grid[m1m["features"]])
print("predicted pace loss (sec) as a tyre ages, at this circuit:")
print(grid.pivot(index="TyreLife", columns="compound_C", values="pred").round(3).to_string())
print("columns are the real compound: C1 hardest, C5 softest")

# ---------- MODEL 3: pit probability through the race ----------
print("\n" + "=" * 68); print("MODEL 3: will he pit next lap?"); print("=" * 68)
q = pit[pit.Round == REPLAY_ROUND].copy()
for c in m3m["categorical"]:
    q[c] = q[c].astype("category").cat.set_categories(pit[c].astype("category").cat.categories)
q["p_pit"] = m3.predict_proba(q[m3m["features"]])[:, 1]
thr = m3m["threshold"]
dq = q[q.Driver == drv].sort_values("LapNumber")
stops = dq[dq.pit_next_lap == 1].LapNumber.tolist()
print(f"{drv} actually pitted at the end of lap(s): {stops} | alert threshold {thr:.2f}")
print("\nlap-by-lap around each real stop (5 laps before, 1 after):")
show = sorted({l for s in stops for l in range(s-5, s+2)} & set(dq.LapNumber))
print(dq[dq.LapNumber.isin(show)][["LapNumber","Compound","TyreLife","gap_ahead","sc_or_vsc","p_pit","pit_next_lap"]]
       .assign(alert=lambda x: np.where(x.p_pit >= thr, "ALERT", ""))
       .round(3).to_string(index=False))
hit = ((q.p_pit >= thr) & (q.pit_next_lap == 1)).sum()
print(f"\nacross the whole race: {int(q.pit_next_lap.sum())} stops, model flagged {int((q.p_pit>=thr).sum())} laps, "
      f"caught {int(hit)} of them")
print("Most alerts are false. The model narrows where to watch; it does not decide strategy.")

# ---------- EXPORT for the website: round replay ----------
OUT = os.path.join(BASE, "site_export"); os.makedirs(OUT, exist_ok=True)
p[["Driver","Team","LapNumber","Compound","TyreLife","lap_delta","pred_delta"]].rename(
    columns={"lap_delta":"actual_vs_own_pace","pred_delta":"predicted"}).to_csv(os.path.join(OUT,"replay_model1_laps.csv"), index=False)

cs = sorted(p.compound_C.dropna().unique())
g2 = pd.DataFrame([{"TyreLife": a, "compound_C": c} for c in cs for a in range(1, 46)])
g2["laps_left"] = p.laps_left.median(); g2["gap_ahead"] = 3.0; g2["dirty_air"] = 0
g2["team_fp_deg"] = p.team_fp_deg.median(); g2["team_longrun_gap_pct"] = p.team_longrun_gap_pct.median()
for c, src in [("Compound", p.Compound), ("Team", p.Team), ("circuit_type", p.circuit_type)]:
    g2[c] = pd.Categorical([src.mode().iloc[0]] * len(g2), categories=src.cat.categories)
g2["pred_pace_loss_sec"] = m1.predict(g2[m1m["features"]])
mx = pace[pace.Round <= 16].groupby("compound_C").TyreLife.max()
g2["seen_in_training"] = (g2.TyreLife <= g2.compound_C.map(mx)).astype(int)
g2[["compound_C","TyreLife","pred_pace_loss_sec","seen_in_training"]].to_csv(os.path.join(OUT,"replay_model2_degradation.csv"), index=False)

q[["Driver","LapNumber","Compound","TyreLife","p_pit","pit_next_lap"]].assign(alert=(q.p_pit >= thr).astype(int)) \
 .to_csv(os.path.join(OUT,"replay_model3_pit.csv"), index=False)

json.dump({"round": REPLAY_ROUND, "event": str(ev), "followed_driver": str(drv), "pit_threshold": float(thr),
           "pit_model": "model3_pit.pkl", "trained_on_rounds": "1-16", "laps_replayed": int(len(p)),
           "mae_sec": round(float(err.mean()), 3), "median_abs_err_sec": round(float(err.median()), 3),
           "stops": int(q.pit_next_lap.sum()), "laps_flagged": int((q.p_pit >= thr).sum()), "stops_caught": int(hit)},
          open(os.path.join(OUT,"replay_meta.json"), "w"), indent=2)
print("saved to", OUT)

replaying: United States Grand Prix (round 18) | models trained on rounds 1-16 only

MODEL 1: race pace
clean green laps replayed: 901 | average error 0.554 sec | half the laps within 0.420 sec

following ANT (Mercedes): predicted vs actual pace, every 8th lap
 LapNumber Compound  TyreLife  actual_vs_own_pace  predicted
         5   MEDIUM       5.0                0.88       0.69
        13   MEDIUM      13.0                1.66       0.60
        21   MEDIUM      21.0                1.01       0.50
        31     HARD       7.0               -0.18      -0.44
        39     HARD      15.0               -0.29      -0.42
        47     HARD      23.0                0.00      -0.41

MODEL 2: tyre degradation, read from the model
predicted pace loss (sec) as a tyre ages, at this circuit:
compound_C    2.0    3.0    4.0
TyreLife                       
1          -0.793 -0.808 -0.678
10         -0.107 -0.122 -0.176
20          0.128  0.113  0.060
30          0.128  0.113  0.060
columns are t